# AvtoPassport 01: Phase 1 smoke test (Colab + local)

**Purpose.** The Phase 1 smoke test from `CLAUDE.md` section 3: sample frames from our walkaround video, filter them by
quality, blur plates and faces, run every candidate damage model, the parts model and a COCO detector on the frames and
photos, read / validate / decode the VIN, and measure latency on GPU and CPU. Pass condition (CLAUDE.md): annotated
images look sensible, the VIN is read correctly or the failure is understood, and latencies are recorded in
`docs/research/smoke_test_results.md` (tables generated by `scripts/render_results.py` from this notebook's outputs).

**Privacy rule.** No pixels are displayed or saved before anonymisation. The anonymiser (licence plates + faces, fail
closed) is built before any pixel is decoded, and every kept frame and photo goes through it before anything is shown
or written; frames rejected by the quality filter are listed, never shown; VIN photos are never shown or saved, and
every VIN string is masked after position 11. `outputs/` is gitignored; executed copies of this notebook go to
`outputs/smoke/<run>/`. Only this cleared notebook (no outputs) is committed. It is generated by
`scripts/build_notebook_01.py`: edit the builder, not the notebook.

**Run matrix**

| Run | Where | Device | Settings |
|---|---|---|---|
| R0 | local | GPU | dry run with small caps: checks the plumbing, numbers are not reported |
| R1 | local, RTX 5050 laptop | `cuda` | full run with `AVTO_LATENCY=1` |
| R2 | local, CUDA hidden | `cpu` | full run with `CUDA_VISIBLE_DEVICES= AVTO_LATENCY=1` |
| Colab GPU | Google Colab, GPU runtime | `cuda` | user step |
| Colab CPU | Google Colab, CPU runtime | `cpu` | user step |

Local runs, from the repository root (the executed copy is written next to the run's outputs):

```bash
# R0 dry run
AVTO_RUN=R0-dry AVTO_MAX_FRAMES=4 AVTO_MAX_IMAGES=1 AVTO_MAX_VINS=1 AVTO_LATENCY=1 AVTO_LAT_FRAMES=2 \
AVTO_LAT_WARMUP=1 AVTO_LAT_RUNS=2 AVTO_LAT_CAP_S=20 .venv/bin/jupyter nbconvert --to notebook --execute \
  notebooks/01_smoke_test_colab.ipynb --ExecutePreprocessor.timeout=-1 \
  --output-dir outputs/smoke/R0-dry --output 01_smoke_test_colab.executed.ipynb
rm -rf outputs/smoke/R0-dry  # after checking it: scripts/render_results.py renders every folder in outputs/smoke/
# R1: AVTO_RUN=R1-local-gpu AVTO_LATENCY=1 (same command, --output-dir outputs/smoke/R1-local-gpu)
# R2: CUDA_VISIBLE_DEVICES= AVTO_RUN=R2-local-cpu AVTO_LATENCY=1 (--output-dir outputs/smoke/R2-local-cpu)
```

On Colab: open this notebook, choose the runtime type (GPU or CPU), fill `REPO_URL` or `DRIVE_ZIP` and either
`UPLOAD = True` or `DRIVE_RAW_DIR` in cell 1a (optionally `OVERRIDES`), then Runtime > Run all. Step 9 zips the
outputs and offers the download.

**Environment variables** (all optional; on Colab use `OVERRIDES` in cell 1a)

| Variable | Default | Meaning |
|---|---|---|
| `AVTO_RUN` | `local-<device>` (`colab-<device>` on Colab) | run id; outputs go to `outputs/smoke/<AVTO_RUN>/`, replaced on re-run |
| `AVTO_DEVICE` | `cuda` if `torch.cuda.is_available()` else `cpu` | `auto`, `cpu`, `cuda` or `cuda:N` |
| `AVTO_MAX_FRAMES` | none | cap on sampled frames per video (`0` skips the video) |
| `AVTO_MAX_IMAGES` | none | cap on photos from `data/raw/images/` |
| `AVTO_MAX_VINS` | none | cap on VIN photos from `data/raw/vin/` |
| `AVTO_FPS` | `2.5` | frame sampling rate (CLAUDE.md: 2-3 fps) |
| `AVTO_MODELS` | all | comma-separated model keys (step 3) |
| `AVTO_LATENCY` | `0` | `1` runs the latency tiers (step 7) and the estimate (step 8) |
| `AVTO_LAT_TIERS` | `gpu` on CUDA, else `cpu_torch,cpu_ort16,cpu_ort2` | latency tiers (step 7) |
| `AVTO_LAT_FRAMES` | `10` | anonymised frames used for latency |
| `AVTO_LAT_WARMUP` | `3` | untimed warm-up predictions per model and tier |
| `AVTO_LAT_RUNS` | `20` | timed predictions per model and tier |
| `AVTO_LAT_CAP_S` | `120` | cap in seconds on the timed phase per model and tier |
| `AVTO_EASYOCR` | `0` | `1` also runs EasyOCR on the VIN photos as a comparison |

Cell 1a forces `YOLO_AUTOINSTALL=false` and sets `YOLO_OFFLINE=true` (unless already set). `OVERRIDES` accepts
`AVTO_*` keys only.

## 1a. Bootstrap

Forces `YOLO_AUTOINSTALL=false` before anything can import ultralytics (no pip installs at runtime: with auto-install
on, ultralytics' patched `PIL.Image.open` pip-installs `pi-heif` the first time any image fails to open) and sets
`YOLO_OFFLINE=true` unless already set (no ultralytics analytics or update checks; models come from
`scripts/download_models.py`).
Finds the repository root (walks up from the working directory to `pyproject.toml`; on Colab `REPO_DIR`), changes
into it and puts `src/` first on `sys.path`, so `avtopassport` is imported from this checkout.

Colab only: gets the repository (`REPO_URL` clone or `DRIVE_ZIP`), installs `requirements-colab.txt`, keeps a single
OpenCV distribution (the headless build; a runtime restart is needed only if `cv2` then fails to import), and copies
the inputs into `data/raw/{video,images,vin}` (`UPLOAD` file picker, sorted by extension and by "VIN" in the file name,
or `DRIVE_RAW_DIR`).

In [ ]:
# 1a. Bootstrap. YOLO_AUTOINSTALL is forced first: cells 2, 3, 5 and 7 import ultralytics.
import os

# Forced, not a default: with auto-install on, ultralytics pip-installs at runtime (its PIL.Image.open patch
# installs pi-heif the first time any image fails to open).
os.environ["YOLO_AUTOINSTALL"] = "false"
os.environ.setdefault("YOLO_OFFLINE", "true")  # ultralytics: no analytics, update checks or asset downloads

import importlib.metadata as md
import re
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# ---- Colab settings (ignored locally) ----
REPO_URL = ""  # e.g. "https://github.com/<user>/avtopassport.git": cloned into REPO_DIR
REPO_DIR = "/content/avtopassport"  # where the repository lives on Colab
DRIVE_ZIP = ""  # e.g. "/content/drive/MyDrive/avtopassport.zip": a zip of the repository, unpacked into REPO_DIR
DRIVE_RAW_DIR = ""  # e.g. "/content/drive/MyDrive/avtopassport_raw" holding video/ images/ vin/
UPLOAD = False  # True: choose the video, photos and VIN photo in a browser file picker
OVERRIDES: dict[str, str] = {}  # e.g. {"AVTO_LATENCY": "1"}: AVTO_* only, put into os.environ by cell 1c

VIDEO_EXTS = {".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm"}
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".heic", ".heif", ".webp"}
OPENCV_DISTS = {"opencv-python", "opencv-python-headless", "opencv-contrib-python", "opencv-contrib-python-headless"}
ORT_DISTS = {"onnxruntime", "onnxruntime-gpu"}


def run_cmd(cmd: list[str], *, check: bool = True, **kwargs) -> subprocess.CompletedProcess:
    """Run a command, print it and the tail of its output; raise on a non-zero exit if `check`."""
    print("$", " ".join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True, **kwargs)
    tail = (proc.stdout + proc.stderr).strip().splitlines()[-20:]
    if tail:
        print("\n".join(tail))
    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {' '.join(cmd)}")
    return proc


def installed(names: set[str]) -> dict[str, str]:
    """Installed distributions among `names` (normalised names) -> version."""
    found: dict[str, str] = {}
    for dist in md.distributions():
        meta = dist.metadata
        name = ((meta.get("Name") if meta is not None else None) or "").lower().replace("_", "-")
        if name in names:
            found[name] = dist.version
    return found


def find_repo_root(start: Path) -> Path:
    """Walk up from `start` to the first directory holding pyproject.toml."""
    for d in (start, *start.parents):
        if (d / "pyproject.toml").is_file():
            return d
    raise FileNotFoundError(f"no pyproject.toml above {start}: start Jupyter inside the repository")


def upload_subdir(filename: str) -> str | None:
    """data/raw/ sub-folder for an uploaded file: 'vin' (VIN in the name, or ground_truth.csv), 'video', 'images'."""
    p = Path(filename)
    low, ext = p.name.lower(), p.suffix.lower()
    if low == "ground_truth.csv" or (ext in IMAGE_EXTS and re.search(r"(?<![a-z])vin", low)):
        return "vin"
    if ext in VIDEO_EXTS:
        return "video"
    if ext in IMAGE_EXTS:
        return "images"
    return None


if "ultralytics" in sys.modules:
    print("note: ultralytics was imported before this cell; restart the kernel so YOLO_AUTOINSTALL applies")

if IN_COLAB:
    drive_paths = [p for p in (DRIVE_ZIP, DRIVE_RAW_DIR) if p.startswith("/content/drive")]
    if drive_paths and not Path("/content/drive/MyDrive").exists():
        from google.colab import drive

        drive.mount("/content/drive")
    repo = Path(REPO_DIR)
    if not (repo / "pyproject.toml").is_file():
        if REPO_URL:
            run_cmd(["git", "clone", "--depth", "1", REPO_URL, str(repo)])
        elif DRIVE_ZIP:
            unpacked = Path("/content/_avtopassport_unzip")
            shutil.rmtree(unpacked, ignore_errors=True)
            shutil.unpack_archive(DRIVE_ZIP, unpacked)
            roots = sorted((p.parent for p in unpacked.rglob("pyproject.toml")), key=lambda p: len(p.parts))
            if not roots:
                raise FileNotFoundError(f"no pyproject.toml inside {DRIVE_ZIP}")
            shutil.move(str(roots[0]), str(repo))
        else:
            raise FileNotFoundError(f"Colab: set REPO_URL or DRIVE_ZIP, or put the repository at {REPO_DIR}")
    REPO_ROOT = repo.resolve()
else:
    REPO_ROOT = find_repo_root(Path.cwd().resolve())

os.chdir(REPO_ROOT)
SRC = str(REPO_ROOT / "src")
if sys.path[0] != SRC:
    sys.path.insert(0, SRC)

if IN_COLAB:
    numpy_before = installed({"numpy"})
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"])
    # Colab ships opencv-python, -headless and -contrib, which share the cv2/ folder: keep only the headless build.
    cv_dists = installed(OPENCV_DISTS)
    extra_cv = sorted(n for n in cv_dists if n != "opencv-python-headless")
    if extra_cv:
        run_cmd([sys.executable, "-m", "pip", "uninstall", "-y", *extra_cv])
    if extra_cv or "opencv-python-headless" not in cv_dists:
        # uninstalling a sibling deletes shared cv2/ files, so the headless wheel is reinstalled on top
        headless = cv_dists.get("opencv-python-headless")
        spec = f"opencv-python-headless=={headless}" if headless else "opencv-python-headless"
        run_cmd([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps", spec])
    ort_dists = installed(ORT_DISTS)
    if len(ort_dists) > 1:  # the CPU and GPU builds share the onnxruntime/ folder: keep the CPU build
        run_cmd([sys.executable, "-m", "pip", "uninstall", "-y", "onnxruntime-gpu"])
        run_cmd([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
                 f"onnxruntime=={ort_dists['onnxruntime']}"])
    if installed({"numpy"}) != numpy_before and "numpy" in sys.modules:
        print("WARNING: pip changed numpy after it was imported: Runtime > Restart session, then run all again")
    try:
        import cv2

        print("cv2", cv2.__version__)
    except Exception as exc:
        print(f"cv2 import failed after the OpenCV clean-up ({exc!r}). A runtime restart may be needed: "
              "Runtime > Restart session, then run all cells again.")
        raise

import avtopassport
from avtopassport import paths as avto_paths

if Path(avtopassport.__file__).resolve().parent != (REPO_ROOT / "src" / "avtopassport").resolve():
    raise RuntimeError(f"avtopassport was imported from {avtopassport.__file__}, not {SRC}: restart the kernel")
if avto_paths.REPO_ROOT.resolve() != REPO_ROOT:
    raise RuntimeError(f"avtopassport.paths.REPO_ROOT={avto_paths.REPO_ROOT} != {REPO_ROOT} (check AVTO_REPO_ROOT)")
RAW = avto_paths.RAW

if IN_COLAB and DRIVE_RAW_DIR:
    for sub in ("video", "images", "vin"):
        if (Path(DRIVE_RAW_DIR) / sub).is_dir():
            shutil.copytree(Path(DRIVE_RAW_DIR) / sub, RAW / sub, dirs_exist_ok=True)
if IN_COLAB and UPLOAD:
    from google.colab import files

    for name, data in files.upload().items():
        sub = upload_subdir(name)
        if sub is None:
            print("skipped (not a video, image or ground_truth.csv):", name)
            continue
        dest = RAW / sub / Path(name).name
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(data)
        print(f"{name} -> data/raw/{sub}/")

print("REPO_ROOT:", REPO_ROOT, "| Colab:", IN_COLAB, "| YOLO_AUTOINSTALL:", os.environ["YOLO_AUTOINSTALL"])
for sub in ("video", "images", "vin"):
    d = RAW / sub
    n_files = sum(1 for p in d.iterdir() if p.is_file() and not p.name.startswith(".")) if d.is_dir() else 0
    print(f"data/raw/{sub}: {n_files} file(s)")

## 1b. Helpers

Pure functions used by the later cells (nothing runs here): environment-variable parsing, VIN masking (including
validation corrections that name a character after position 11), the unmasked-VIN guard, guarded JSON / CSV /
Markdown writers, statistics, ground-truth matching and OCR reading order (step 6), the per-inspection estimate
(step 8) and the summary renderer (step 9). Every file the notebook writes, except anonymised JPEGs via `save_jpeg`,
goes through `write_text`, which refuses to write any VIN read in step 6 unless it is masked; VIN tables print
through the same guard.

In [ ]:
# 1b. Helpers: pure functions, no pixels, no I/O when this cell runs.
import csv
import io
import json
import math
import os
import re
import statistics
import time
from collections import Counter
from dataclasses import asdict, is_dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Sequence

import numpy as np

# ---- environment variables ----
_TRUE, _FALSE = {"1", "true", "yes", "y", "on"}, {"0", "false", "no", "n", "off"}


def env_str(name: str, default: str) -> str:
    """Stripped value of `name`, or `default` when unset or empty."""
    value = os.environ.get(name, "").strip()
    return value or default


def env_bool(name: str, default: bool = False) -> bool:
    """0/1, true/false, yes/no, on/off; unset or empty -> default; anything else raises."""
    value = os.environ.get(name, "").strip().lower()
    if not value:
        return default
    if value in _TRUE:
        return True
    if value in _FALSE:
        return False
    raise ValueError(f"{name}={value!r}: expected 0/1, true/false, yes/no or on/off")


def env_int(name: str, default: int | None, *, minimum: int = 0) -> int | None:
    """Integer >= minimum; unset, empty, 'none' or 'all' -> default (None means no cap)."""
    value = os.environ.get(name, "").strip().lower()
    if value in {"", "none", "all"}:
        return default
    try:
        number = int(value)
    except ValueError:
        raise ValueError(f"{name}={value!r} is not an integer") from None
    if number < minimum:
        raise ValueError(f"{name}={number} must be >= {minimum}")
    return number


def env_float(name: str, default: float) -> float:
    """Finite float > 0; unset or empty -> default."""
    value = os.environ.get(name, "").strip()
    if not value:
        return default
    try:
        number = float(value)
    except ValueError:
        raise ValueError(f"{name}={value!r} is not a number") from None
    if not math.isfinite(number) or number <= 0:
        raise ValueError(f"{name}={value!r} must be a finite number > 0")
    return number


def env_list(name: str, default: Sequence[str], is_valid: Callable[[str], bool]) -> list[str]:
    """Comma-separated items (order kept, duplicates dropped); unset, empty or 'all' -> default."""
    value = os.environ.get(name, "").strip()
    if not value or value.lower() == "all":
        return list(default)
    items = list(dict.fromkeys(s.strip() for s in value.split(",") if s.strip()))
    bad = [s for s in items if not is_valid(s)]
    if bad or not items:
        raise ValueError(f"{name}={value!r}: unknown item(s) {bad}")
    return items


# ---- VIN masking and the unmasked-VIN guard ----
VIN_SHOWN = 11  # characters kept by mask_vin
SECRET_VINS: set[str] = set()  # every VIN-like string read in step 6 (uppercase); never written or printed
_VIN_TOKEN = re.compile(r"(?<![A-Za-z0-9])([A-Za-z0-9]{11})[A-Za-z0-9]{6}(?![A-Za-z0-9])")


def mask_vin(vin: str | None) -> str | None:
    """First 11 characters + '******'; None / '' unchanged."""
    return vin[:VIN_SHOWN] + "******" if vin else vin


def add_secret(*values: str | None) -> None:
    """Remember strings longer than the masked prefix that must never be written or printed in full."""
    for v in values:
        if isinstance(v, str) and len(v.strip()) > VIN_SHOWN:
            SECRET_VINS.add(v.strip().upper())


def scrub(text: str) -> str:
    """Mask remembered secrets and any free-standing 17-character alphanumeric token."""
    for s in sorted(SECRET_VINS, key=len, reverse=True):
        text = re.sub(re.escape(s), lambda _m, masked=mask_vin(s): masked, text, flags=re.IGNORECASE)
    return _VIN_TOKEN.sub(lambda m: m.group(1) + "******", text)


_TAIL_FIX = re.compile(r"\bpos (\d+): (\S)->(\S)")  # avtopassport.vin.validate corrections, e.g. 'pos 16: S->5'


def redact_tail(text: str) -> str:
    """Hide corrections that name a character after position 11 ('pos 16: S->5' -> 'pos 16: *->*')."""
    return _TAIL_FIX.sub(lambda m: m.group(0) if int(m.group(1)) <= VIN_SHOWN else f"pos {m.group(1)}: *->*", text)


def scrub_obj(obj: Any) -> Any:
    """scrub() every string (keys included) inside a JSON-like structure."""
    if isinstance(obj, str):
        return scrub(obj)
    if isinstance(obj, dict):
        return {scrub(str(k)): scrub_obj(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [scrub_obj(v) for v in obj]
    return obj


def guard(text: str) -> str:
    """Return `text`, or raise if it contains a remembered unmasked VIN (case-insensitive)."""
    upper = text.upper()
    if any(s in upper for s in SECRET_VINS):
        raise RuntimeError("refusing to write or print an unmasked VIN")
    return text


def exc_text(exc: BaseException, limit: int = 400) -> str:
    """One-line 'Type: message', VIN-scrubbed and truncated."""
    text = scrub(" ".join(f"{type(exc).__name__}: {exc}".split()))
    return text if len(text) <= limit else text[: limit - 3] + "..."


ERRORS: list[dict[str, Any]] = []


def record_error(stage: str, exc: BaseException | str, **ctx: Any) -> None:
    """Record a one-line error (VIN-scrubbed) in ERRORS and print it; the run continues."""
    msg = scrub(exc) if isinstance(exc, str) else exc_text(exc)
    row = {"stage": stage, **{k: scrub(str(v)) for k, v in ctx.items()}, "error": msg}
    ERRORS.append(row)
    print(guard(f"[error] {row}"))


# ---- writers (all guarded) ----
def _json_default(o: Any) -> Any:
    if is_dataclass(o) and not isinstance(o, type):
        return asdict(o)
    if isinstance(o, np.generic):
        return o.item()
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, Path):
        return o.as_posix()
    if isinstance(o, (set, frozenset)):
        return sorted(o)
    raise TypeError(f"not JSON serialisable: {type(o).__name__}")


def write_text(path: Path, text: str) -> Path:
    """Write UTF-8 text after the unmasked-VIN guard; creates parent folders."""
    guard(text)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
    return path


def write_json(path: Path, obj: Any) -> Path:
    return write_text(path, json.dumps(obj, indent=2, ensure_ascii=False, default=_json_default) + "\n")


def csv_text(rows: Sequence[dict], fields: Sequence[str]) -> str:
    buf = io.StringIO()
    writer = csv.DictWriter(buf, fieldnames=list(fields), restval="", extrasaction="ignore", lineterminator="\n")
    writer.writeheader()
    writer.writerows(rows)
    return buf.getvalue()


def write_csv(path: Path, rows: Sequence[dict], fields: Sequence[str]) -> Path:
    return write_text(path, csv_text(rows, fields))


def read_csv(path: Path) -> list[dict[str, str]]:
    """Rows of a CSV file ([] if it does not exist); a UTF-8 byte-order mark is ignored."""
    if not path.is_file():
        return []
    with path.open(newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))


def append_csv(path: Path, rows: Sequence[dict], fields: Sequence[str]) -> Path:
    """Append rows; an existing file keeps its header order and missing `fields` are added at the end."""
    header: list[str] = []
    if path.is_file():
        with path.open(newline="", encoding="utf-8-sig") as f:
            header = next(csv.reader(f), [])
    header += [c for c in fields if c not in header]
    return write_csv(path, read_csv(path) + list(rows), header)


# ---- numbers ----
def med(xs: Sequence[float]) -> float | None:
    return float(statistics.median(xs)) if len(xs) else None


def mean(xs: Sequence[float]) -> float | None:
    return float(statistics.fmean(xs)) if len(xs) else None


def pctl(xs: Sequence[float], q: float) -> float | None:
    """Percentile q in [0, 1] with linear interpolation (numpy's default); None without samples."""
    return float(np.percentile(np.asarray(xs, dtype=float), q * 100)) if len(xs) else None


def rnd(x: float | None, nd: int = 3) -> float | None:
    return None if x is None else round(float(x), nd)


def f3(x: float | None) -> str:
    """CSV number format shared with scripts/bench_cpu.py: 3 decimals, '' for None."""
    return "" if x is None else f"{x:.3f}"


def f2(x: float | None) -> str:
    """CSV load-average format shared with scripts/bench_cpu.py: 2 decimals, '' for None."""
    return "" if x is None else f"{x:.2f}"


def to_float(value: Any) -> float | None:
    try:
        number = float(value)
    except (TypeError, ValueError):
        return None
    return number if math.isfinite(number) else None


def hamming(a: str | None, b: str | None) -> int | None:
    """Character errors between two 17-character strings; None if either is not 17 characters."""
    if not a or not b or len(a) != 17 or len(b) != 17:
        return None
    return sum(x != y for x, y in zip(a, b))


def evenly_spaced(n: int, k: int) -> list[int]:
    """Up to k distinct indices spread evenly over range(n) (k == 1 -> the middle)."""
    if n <= 0 or k <= 0:
        return []
    if k >= n:
        return list(range(n))
    if k == 1:
        return [n // 2]
    return sorted({int(i * (n - 1) / (k - 1) + 0.5) for i in range(k)})


def unique_name(name: str, used: set[str]) -> str:
    """`name`, or `name_2`, `name_3`, ... if already used; records the result in `used`."""
    out, i = name, 1
    while out in used:
        i += 1
        out = f"{name}_{i}"
    used.add(out)
    return out


def now_iso() -> str:
    return datetime.now(timezone.utc).astimezone().isoformat(timespec="seconds")


# ---- VIN photos: ground truth and OCR reading order (step 6) ----
def ground_truth_map(rows: Sequence[dict[str, Any]]) -> dict[str, str]:
    """ground_truth.csv rows (columns file, vin; header case ignored) -> {lower-case file name: VIN}."""
    out: dict[str, str] = {}
    for row in rows:
        row = {str(k or "").strip().lower(): v for k, v in row.items()}
        name = Path(str(row.get("file") or "").strip()).name.lower()
        vin = re.sub(r"\s", "", str(row.get("vin") or "")).upper()
        if name and vin:
            out[name] = vin
    return out


def gt_key(gt: dict[str, str], file_name: str) -> str | None:
    """The ground-truth entry for a VIN photo: same file name, else same stem (case-insensitive)."""
    name = Path(file_name).name.lower()
    if name in gt:
        return name
    stem = Path(name).stem
    return next((k for k in gt if Path(k).stem == stem), None)


def reading_order(boxes: Sequence[Any]) -> list[int]:
    """Indices of OCR text boxes (corner points) in reading order: rows top to bottom (centres within half a box
    height share a row), left to right within a row."""
    pts = [np.asarray(b, dtype=float).reshape(-1, 2) for b in boxes]
    cy = [float(p[:, 1].mean()) for p in pts]
    x0 = [float(p[:, 0].min()) for p in pts]
    hs = [float(p[:, 1].max() - p[:, 1].min()) for p in pts]
    rows: list[list[int]] = []
    row_cy = row_h = 0.0
    for i in sorted(range(len(pts)), key=lambda j: cy[j]):
        if rows and abs(cy[i] - row_cy) <= 0.5 * max(hs[i], row_h):
            rows[-1].append(i)
        else:
            rows.append([i])
            row_cy, row_h = cy[i], hs[i]
    return [i for row in rows for i in sorted(row, key=lambda j: x0[j])]


# ---- per-inspection estimate (step 8) ----
EST_FIELDS = ["run", "tier", "step", "included", "count", "median_ms", "subtotal_ms", "source", "basis",
              "x2 safety factor (assumption)", "note"]
EST_BASIS = "derived from measured medians"
ESTIMATE_STEPS = (  # (label, kind, StepTimer step or model key, count key)
    ("video probe", "timer", "video.probe", "videos"),
    ("video decode + frame sampling", "timer", "frames.extract", "videos"),
    ("quality filter", "timer", "quality.filter", "videos"),
    ("privacy on kept frames (plates, faces, parts plate class)", "timer", "privacy.frame", "kept_frames"),
    ("photo decode", "timer", "image.load", "photos"),
    ("privacy on photos", "timer", "privacy.photo", "photos"),
    ("damage segmentation damage_mitbersh", "model", "damage_mitbersh", "model_inputs"),
    ("part segmentation parts_mitbersh", "model", "parts_mitbersh", "model_inputs"),
    ("VIN photo decode", "timer", "vin.load", "vins"),
    ("VIN OCR (RapidOCR)", "timer", "vin.ocr", "vins"),
    ("VIN validation", "timer", "vin.validate", "vins"),
    ("VIN decode (WMI table + NHTSA vPIC)", "timer", "vin.decode", "vins"),
)
ESTIMATE_EXCLUDED = (
    ("model loading, ONNX export, warm-up", "one-off per server start, not per inspection"),
    ("damage_danishali, damage_harpreet", "alternative damage candidates, not additive"),
    ("coco_yolo26n", "sanity check, not part of the pipeline"),
    ("plate_n latency rows", "the plate detector is already inside the privacy.* StepTimer medians"),
    ("raw-ort latency rows", "inference only; the end-to-end (e2e) rows are used"),
    ("EasyOCR", "comparison only"),
    ("annotated images, contact sheets, JPEG writes", "smoke-test reporting only"),
    ("mapping, dedupe, severity, cost, passport store / PDF", "not implemented in Phase 1"),
    ("upload and network transfer to the server", "not measured"),
)


def _tier_order(tier: str) -> tuple[int, int, str]:
    m = re.fullmatch(r"cpu_ort(\d+)", tier)
    return ({"gpu": 0, "cpu_torch": 1}.get(tier, 2), -int(m.group(1)) if m else 0, tier)


def build_estimate(run: str, lat_rows: Sequence[dict], timer_summary: dict, counts: dict, *,
                   device: str, cpu_count: int | None = None) -> list[dict]:
    """Per tier: sum over ESTIMATE_STEPS of (measured median x count in this run). Model medians come from the
    tier's end-to-end latency rows (protocol 'e2e...', no error); other steps from this run's StepTimer, measured in
    this process (on `device`, with the libraries' default thread pools on `cpu_count` CPUs), which the notes state."""
    rows = [r for r in lat_rows if r.get("run") == run and r.get("tier")]
    out: list[dict] = []
    for tier in sorted({r["tier"] for r in rows}, key=_tier_order):
        model_ms: dict[str, float] = {}
        for r in rows:
            ms = to_float(r.get("median_ms"))
            if (r["tier"] == tier and not (r.get("error") or "").strip()
                    and str(r.get("protocol", "")).startswith("e2e") and ms is not None):
                model_ms.setdefault(r.get("model", ""), ms)
        safety = tier == "cpu_ort2"
        total, missing = 0.0, []
        for label, kind, key, count_key in ESTIMATE_STEPS:
            n = int(counts.get(count_key) or 0)
            if kind == "model":
                median = model_ms.get(key)
                source = f"latency.csv tier={tier} model={key} median_ms (e2e)"
            else:
                median = (timer_summary.get(key) or {}).get("median_ms")
                source = f"StepTimer {key} median (this run, device={device})"
            subtotal = 0.0 if n == 0 else (None if median is None else median * n)
            if subtotal is None:
                missing.append(label)
            else:
                total += subtotal
            out.append({"run": run, "tier": tier, "step": label, "included": True, "count": n,
                        "median_ms": f3(median), "subtotal_ms": f3(subtotal), "source": source, "basis": EST_BASIS,
                        "x2 safety factor (assumption)": f3(2 * subtotal) if safety and subtotal is not None else "",
                        "note": "no measurement in this run" if subtotal is None else ""})
        notes = [f"incomplete: missing {', '.join(missing)}"] if missing else []
        tier_threads = int(tier[len("cpu_ort"):]) if re.fullmatch(r"cpu_ort\d+", tier) else None
        if tier != "gpu" and device.startswith("cuda"):
            notes.append(f"non-model steps were measured on {device} in this run; use the CPU run for a CPU deploy")
        elif tier_threads and cpu_count and tier_threads < cpu_count:
            notes.append(f"non-model steps (decode, privacy, OCR, vPIC) ran in the notebook process with default "
                         f"thread pools on {cpu_count} CPUs, not {tier_threads} threads: optimistic for this tier")
        out.append({"run": run, "tier": tier, "step": "TOTAL per inspection (included steps)", "included": True,
                    "count": "", "median_ms": "", "subtotal_ms": f3(total), "source": "sum of the rows above",
                    "basis": EST_BASIS, "x2 safety factor (assumption)": f3(2 * total) if safety else "",
                    "note": "; ".join(notes)})
    if out:
        for label, why in ESTIMATE_EXCLUDED:
            out.append({"run": run, "tier": "(all)", "step": label, "included": False, "count": "", "median_ms": "",
                        "subtotal_ms": "", "source": "", "basis": "", "x2 safety factor (assumption)": "",
                        "note": f"excluded: {why}"})
    return out


# ---- text tables ----
def md_cell(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, float):
        return f"{value:,.1f}" if abs(value) >= 10 else f"{value:.3f}"
    if isinstance(value, (list, tuple)):
        value = ", ".join(str(v) for v in value)
    elif isinstance(value, dict):
        value = ", ".join(f"{k}: {v}" for k, v in value.items())
    return str(value).replace("|", "\\|").replace("\n", " ")


def md_table(rows: Sequence[dict], cols: Sequence[str]) -> str:
    """GitHub markdown table ('_none_' for no rows)."""
    if not rows:
        return "_none_\n"
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    lines += ["| " + " | ".join(md_cell(r.get(c)) for c in cols) + " |" for r in rows]
    return "\n".join(lines) + "\n"


def print_table(rows: Sequence[dict], cols: Sequence[str]) -> None:
    """Plain-text table through the unmasked-VIN guard (used before anything may be displayed)."""
    if not rows:
        print("(none)")
        return
    cells = [[md_cell(r.get(c)) for c in cols] for r in rows]
    widths = [max(len(str(c)), *(len(row[i]) for row in cells)) for i, c in enumerate(cols)]
    lines = ["  ".join(str(c).ljust(w) for c, w in zip(cols, widths))]
    lines += ["  ".join(v.ljust(w) for v, w in zip(row, widths)) for row in cells]
    print(guard("\n".join(lines)))


def render_summary_md(res: dict) -> str:
    """summary.md for one run: generated markdown tables only."""
    env, gpu, cpu = res.get("env", {}), res.get("env", {}).get("gpu", {}), res.get("env", {}).get("cpu", {})
    versions = env.get("versions", {})
    out = [f"# Smoke test run `{res.get('run')}`", "",
           f"Generated by `notebooks/01_smoke_test_colab.ipynb` on {res.get('created_at')}. Every number is computed "
           "by the notebook in this run. Confidence and quality thresholds are uncalibrated defaults. VINs are masked "
           "after position 11.", "", "## Environment", ""]
    env_rows = [{"item": k, "value": v} for k, v in (
        ("device", gpu.get("device")), ("gpu", gpu.get("name")), ("capability", gpu.get("capability")),
        ("driver", gpu.get("driver")), ("cpu", cpu.get("model")), ("cpu count", cpu.get("count")),
        ("power profile", cpu.get("power_profile")), ("load average", cpu.get("loadavg")),
        ("colab", env.get("colab")), *versions.items())]
    out += [md_table(env_rows, ["item", "value"]), "## Models", ""]
    out += [md_table(res.get("models", []), ["key", "role", "status", "task", "n_classes", "imgsz", "conf", "iou",
                                              "conf_source", "size_bytes", "sha256_ok", "error"]), "## Counts", ""]
    out += [md_table([{"count": k, "value": v} for k, v in res.get("counts", {}).items()], ["count", "value"])]
    quality = res.get("quality", {})
    out += ["## Quality filter", "", md_table([{"item": "thresholds (uncalibrated default)",
                                                "value": quality.get("thresholds")},
                                               {"item": "frames", "value": quality.get("n_frames")},
                                               {"item": "kept", "value": quality.get("n_ok")},
                                               {"item": "reject reasons", "value": quality.get("reasons")}],
                                              ["item", "value"])]
    privacy = res.get("privacy", {})
    out += ["## Privacy", "", md_table([{k: privacy.get(k) for k in ("device", "n_inputs", "n_ok", "n_dropped",
                                                                      "n_plates", "n_faces", "extra_detectors")}],
                                       ["device", "n_inputs", "n_ok", "n_dropped", "n_plates", "n_faces",
                                        "extra_detectors"])]
    out += ["## Detections per model", "", md_table(res.get("detections", []), [
        "model", "device", "inputs", "ok", "errors", "detections", "mean_conf", "e2e_median_ms", "wall_median_ms",
        "max_mem_mb", "per_class"])]
    mapping = res.get("mapping_preview", {})
    out += ["## Damage -> part mapping preview (Phase 3 implements mapping)", "",
            md_table(mapping.get("pairs", []), ["damage_class", "part_class", "n"])]
    vin = res.get("vin", {})
    vin_rows = list(vin.get("results", [])) + list(vin.get("easyocr_results", []))
    out += ["## VIN (masked)", "", md_table(vin_rows, ["file", "engine", "status", "vin_masked", "validation_status",
                                                       "check_digit_ok", "exact_match", "char_errors"])]
    out += ["## Latency", "", md_table(res.get("latency", []), [
        "tier", "model", "imgsz", "device", "threads", "pinned", "n_runs", "median_ms", "p90_ms", "infer_median_ms",
        "wall_median_ms", "cap_hit", "error"])]
    out += ["## Per-inspection estimate (derived from measured medians)", "", md_table(res.get("estimate", []), [
        "tier", "step", "included", "count", "median_ms", "subtotal_ms", "x2 safety factor (assumption)", "note"])]
    timing_rows = [{"step": k, **v} for k, v in sorted(res.get("timings", {}).items())]
    out += ["## Step timings (StepTimer)", "", md_table(timing_rows, ["step", "count", "median_ms", "total_ms",
                                                                      "max_ms"])]
    out += ["## Errors", "", md_table(res.get("errors", []), ["stage", "error"]), "## Artifacts", "",
            "\n".join(f"- `{a}`" for a in res.get("artifacts", [])) or "_none_", ""]
    return "\n".join(out)

## 1c. Run configuration

Reads the `AVTO_*` variables (after applying `OVERRIDES`), picks the device, creates `outputs/smoke/<run>/` (`OUT`;
an existing folder for the same run id is replaced) and the `StepTimer` that times every pipeline step. Invalid values
stop the run here, before any work is done.

In [ ]:
# 1c. Run configuration: AVTO_* variables, device, output folder, step timer.
for key, value in OVERRIDES.items():
    if not str(key).startswith("AVTO_"):
        raise ValueError(f"OVERRIDES: {key!r} is not an AVTO_* setting")
    os.environ[key] = str(value)

from avtopassport.paths import MODELS, OUTPUTS
from avtopassport.timing import StepTimer

try:
    import torch

    CUDA_OK = bool(torch.cuda.is_available())
except Exception as exc:  # torch missing or broken: CPU only
    torch, CUDA_OK = None, False
    print("torch is not importable, CPU only:", exc)

ALL_MODEL_KEYS = ("damage_mitbersh", "damage_danishali", "damage_harpreet", "parts_mitbersh", "coco_yolo26n")

_device = env_str("AVTO_DEVICE", "auto").lower()
DEVICE = ("cuda" if CUDA_OK else "cpu") if _device == "auto" else ("cuda" if _device == "gpu" else _device)
if not re.fullmatch(r"cpu|cuda(:\d+)?", DEVICE):
    raise ValueError(f"AVTO_DEVICE={_device!r}: expected auto, cpu, cuda or cuda:N")
if DEVICE.startswith("cuda") and not CUDA_OK:
    raise ValueError(f"AVTO_DEVICE={_device!r} but torch.cuda.is_available() is False")
IS_CUDA = DEVICE.startswith("cuda")

RUN = env_str("AVTO_RUN", f"{'colab' if IN_COLAB else 'local'}-{DEVICE.replace(':', '')}")
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]{0,63}", RUN):
    raise ValueError(f"AVTO_RUN={RUN!r}: use letters, digits, '.', '_' or '-' (max 64, starting alphanumeric)")
MAX_FRAMES = env_int("AVTO_MAX_FRAMES", None)
MAX_IMAGES = env_int("AVTO_MAX_IMAGES", None)
MAX_VINS = env_int("AVTO_MAX_VINS", None)
FPS = env_float("AVTO_FPS", 2.5)
_models = env_list("AVTO_MODELS", ALL_MODEL_KEYS, lambda k: k in ALL_MODEL_KEYS)
SELECTED_MODELS = [k for k in ALL_MODEL_KEYS if k in _models]
LATENCY = env_bool("AVTO_LATENCY", False)
LAT_TIERS = env_list("AVTO_LAT_TIERS", ["gpu"] if IS_CUDA else ["cpu_torch", "cpu_ort16", "cpu_ort2"],
                     lambda t: t in {"gpu", "cpu_torch"} or re.fullmatch(r"cpu_ort[1-9]\d*", t) is not None)
LAT_FRAMES = env_int("AVTO_LAT_FRAMES", 10, minimum=1)
LAT_WARMUP = env_int("AVTO_LAT_WARMUP", 3, minimum=0)
LAT_RUNS = env_int("AVTO_LAT_RUNS", 20, minimum=1)
LAT_CAP_S = env_float("AVTO_LAT_CAP_S", 120.0)
EASYOCR = env_bool("AVTO_EASYOCR", False)

CONFIG = {
    "AVTO_RUN": RUN, "AVTO_DEVICE": DEVICE, "AVTO_MAX_FRAMES": MAX_FRAMES, "AVTO_MAX_IMAGES": MAX_IMAGES,
    "AVTO_MAX_VINS": MAX_VINS, "AVTO_FPS": FPS, "AVTO_MODELS": SELECTED_MODELS, "AVTO_LATENCY": LATENCY,
    "AVTO_LAT_TIERS": LAT_TIERS, "AVTO_LAT_FRAMES": LAT_FRAMES, "AVTO_LAT_WARMUP": LAT_WARMUP,
    "AVTO_LAT_RUNS": LAT_RUNS, "AVTO_LAT_CAP_S": LAT_CAP_S, "AVTO_EASYOCR": EASYOCR,
    "YOLO_AUTOINSTALL": os.environ.get("YOLO_AUTOINSTALL"), "YOLO_OFFLINE": os.environ.get("YOLO_OFFLINE"),
}
if os.environ.get("YOLO_AUTOINSTALL", "").strip().lower() not in _FALSE:
    raise RuntimeError("YOLO_AUTOINSTALL is not false: run cell 1a first (ultralytics must never pip-install)")

SMOKE_DIR = OUTPUTS / "smoke"
OUT = SMOKE_DIR / RUN
if OUT.exists():  # one run id = one execution; everything inside is regenerated (outputs/ is gitignored)
    shutil.rmtree(OUT)
OUT.mkdir(parents=True)
TIMER = StepTimer()
print_table([{"setting": k, "value": v} for k, v in CONFIG.items()], ["setting", "value"])
print("outputs ->", OUT.relative_to(REPO_ROOT))

## 2. Environment check

Versions (python, torch, torchvision, cv2, ultralytics, onnxruntime, rapidocr, numpy), CUDA availability, GPU name,
compute capability and `torch.cuda.get_arch_list()`; installed OpenCV distributions (exactly one locally, a warning on
Colab) and exactly one of onnxruntime / onnxruntime-gpu; CPU model, `os.cpu_count()`, load average and power profile
(`powerprofilesctl get` if available); a YuNet face-detector smoke test on a blank frame with the model file chosen
by the OpenCV major version. Written to `OUT/env.json`.

In [ ]:
# 2. Environment check -> OUT/env.json
import importlib
import platform

import cv2

from avtopassport.pipeline.privacy import YUNET_V4, YUNET_V5

IMPORT_ERRORS: dict[str, str] = {}


def module_version(name: str) -> str | None:
    """Version of an importable module (__version__, else its distribution's); None if it does not import."""
    try:
        mod = importlib.import_module(name)
    except Exception as exc:
        IMPORT_ERRORS[name] = exc_text(exc)
        return None
    version = getattr(mod, "__version__", None)
    if not version:
        try:
            version = md.version(name)
        except md.PackageNotFoundError:
            version = "unknown"
    return str(version)


def first_value(path: str, prefix: str) -> str | None:
    """Value after ':' on the first line of `path` starting with `prefix` (e.g. /proc/cpuinfo)."""
    try:
        with open(path, encoding="utf-8") as f:
            for line in f:
                if line.startswith(prefix):
                    return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return None


def command_output(cmd: list[str]) -> str | None:
    if not shutil.which(cmd[0]):
        return None
    try:
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=15)
    except (OSError, subprocess.SubprocessError):
        return None
    return (proc.stdout.strip() or None) if proc.returncode == 0 else None


def yunet_smoke() -> dict[str, Any]:
    """FaceDetectorYN with the file for this OpenCV major version, run on a blank (synthetic) frame."""
    major = int(cv2.__version__.split(".")[0])
    path = YUNET_V5 if major >= 5 else YUNET_V4
    rec: dict[str, Any] = {"opencv_major": major, "file": path.relative_to(REPO_ROOT).as_posix()}
    if not path.is_file():
        return {**rec, "status": "pending", "note": "model file missing; step 3 downloads it and retries"}
    try:
        detector = cv2.FaceDetectorYN.create(str(path), "", (320, 240), 0.5, 0.3, 5000)
        detector.setInputSize((320, 240))
        _, faces = detector.detect(np.zeros((240, 320, 3), np.uint8))
        return {**rec, "status": "ok", "faces_on_blank_frame": 0 if faces is None else int(len(faces))}
    except Exception as exc:
        return {**rec, "status": "failed", "error": exc_text(exc)}


versions = {"python": platform.python_version()}
for mod_name in ("torch", "torchvision", "cv2", "ultralytics", "onnxruntime", "rapidocr", "numpy"):
    versions[mod_name] = module_version(mod_name)

gpu: dict[str, Any] = {"device": DEVICE, "cuda_available": CUDA_OK}
if CUDA_OK:
    gpu_index = int(DEVICE.split(":")[1]) if ":" in DEVICE else 0
    capability = torch.cuda.get_device_capability(gpu_index)
    arch_list = torch.cuda.get_arch_list()
    gpu.update(name=torch.cuda.get_device_name(gpu_index), capability=f"{capability[0]}.{capability[1]}",
               arch_list=arch_list, arch_supported=f"sm_{capability[0]}{capability[1]}" in arch_list,
               torch_cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version(),
               memory_gb=round(torch.cuda.get_device_properties(gpu_index).total_memory / 2**30, 2),
               driver=command_output(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"]))
    if not gpu["arch_supported"]:
        print(f"WARNING: sm_{capability[0]}{capability[1]} is not in torch.cuda.get_arch_list() {arch_list}")

try:
    loadavg = [round(x, 2) for x in os.getloadavg()]
except (AttributeError, OSError):
    loadavg = None
cpu = {"model": first_value("/proc/cpuinfo", "model name") or platform.processor() or platform.machine(),
       "count": os.cpu_count(), "loadavg": loadavg, "power_profile": command_output(["powerprofilesctl", "get"]),
       "mem_total": first_value("/proc/meminfo", "MemTotal"),
       "torch_threads": torch.get_num_threads() if torch is not None else None}

cv_dists, ort_dists = installed(OPENCV_DISTS), installed(ORT_DISTS)
if len(cv_dists) != 1:
    if not IN_COLAB:
        raise AssertionError(f"expected exactly one OpenCV distribution, found {cv_dists}")
    print(f"WARNING: {len(cv_dists)} OpenCV distributions installed: {cv_dists}")
assert len(ort_dists) == 1, f"expected exactly one of onnxruntime / onnxruntime-gpu, found {ort_dists}"

try:
    from ultralytics.utils import AUTOINSTALL as ULTRALYTICS_AUTOINSTALL
except Exception:
    ULTRALYTICS_AUTOINSTALL = None
if ULTRALYTICS_AUTOINSTALL:
    raise RuntimeError("ultralytics was imported with auto-install on (before cell 1a?): restart the kernel, run all")

ENV = {"run": RUN, "created_at": now_iso(), "colab": IN_COLAB, "platform": platform.platform(),
       "versions": versions, "import_errors": IMPORT_ERRORS, "gpu": gpu, "cpu": cpu,
       "opencv_distributions": cv_dists, "onnxruntime_distributions": ort_dists,
       "ultralytics_autoinstall": ULTRALYTICS_AUTOINSTALL, "yunet": yunet_smoke()}
write_json(OUT / "env.json", ENV)
print_table([{"item": k, "value": v} for k, v in versions.items()], ["item", "value"])
print_table([{"item": k, "value": v} for k, v in {**gpu, **cpu}.items() if k != "arch_list"], ["item", "value"])
print("OpenCV:", cv_dists, "| ONNX Runtime:", ort_dists, "| YuNet:", ENV["yunet"])

## 3. Models

Downloads any model file missing from `models/manifest.json` with `scripts/download_models.py` (pinned revisions,
sha256-verified), re-checks each file's sha256 against the manifest, then loads every model selected by `AVTO_MODELS`
with ultralytics. A model that fails to load is recorded as `load_failed` with its error and the run continues.
`conf` / `iou` come from the model's own inference config where it ships one (mitbersh); otherwise they are
ultralytics' defaults (0.25 / 0.7), i.e. **uncalibrated defaults** until Phase 2. The COCO model only reports
car / truck / bus / person.

The privacy step always gets its own instance of the parts model (its `license-plate` class is an extra plate
detector), even when `AVTO_MODELS` leaves the parts model out of the inference set; if that instance cannot be loaded,
step 4 stops the run (fail closed).

In [ ]:
# 3. Models: download what is missing (sha256-verified), load every selected candidate, record status.
from ultralytics import YOLO

from avtopassport.io import sha256_file

MANIFEST = MODELS / "manifest.json"
DEFAULT_CONF, DEFAULT_IOU = 0.25, 0.7  # uncalibrated defaults (ultralytics' own); calibrated in Phase 2
MODEL_SPECS: dict[str, dict[str, Any]] = {
    "damage_mitbersh": {"role": "damage", "file": "models/damage/mitbersh_damage_segmentation.pt", "imgsz": 896,
                        "config": "models/damage/mitbersh_damage_inference_config.json",
                        "config_key": "damage_mitbersh_config"},
    "damage_danishali": {"role": "damage", "file": "models/damage/danishali_best.pt", "imgsz": 640},
    "damage_harpreet": {"role": "damage", "file": "models/damage/harpreet_best.pt", "imgsz": 1024},
    "parts_mitbersh": {"role": "parts", "file": "models/parts/mitbersh_parts_segmentation.pt", "imgsz": 768,
                       "config": "models/parts/mitbersh_parts_inference_config.json",
                       "config_key": "parts_mitbersh_config"},
    "coco_yolo26n": {"role": "coco", "file": "models/coco/yolo26n.pt", "imgsz": 640,
                     "report_classes": ("car", "truck", "bus", "person")},
}
assert set(MODEL_SPECS) == set(ALL_MODEL_KEYS)
PRIVACY_FILES = ("plate_n_pt", "plate_n_onnx", "face_yunet_2026may", "face_yunet_2023mar")


def manifest_files() -> dict[str, dict[str, Any]]:
    try:
        return json.loads(MANIFEST.read_text(encoding="utf-8")).get("files", {})
    except FileNotFoundError:
        return {}
    except (OSError, ValueError) as exc:
        record_error("models", exc, file="models/manifest.json")
        return {}


def have(files: dict[str, dict[str, Any]], key: str) -> bool:
    local = (files.get(key) or {}).get("local")
    return bool(local) and (REPO_ROOT / local).is_file()


def inference_config(rel: str | None) -> dict[str, Any]:
    """The 'inference' block of a model's own config JSON ({} if absent)."""
    if not rel:
        return {}
    try:
        return json.loads((REPO_ROOT / rel).read_text(encoding="utf-8")).get("inference") or {}
    except FileNotFoundError:
        return {}
    except (OSError, ValueError) as exc:
        record_error("models", exc, file=rel)
        return {}


def check_sha256(key: str, path: Path, files: dict[str, dict[str, Any]]) -> bool | None:
    """True / False against the manifest; None if the manifest has no hash."""
    expected = (files.get(key) or {}).get("sha256")
    if not expected:
        return None
    with TIMER.step("model.sha256"):
        return sha256_file(path) == expected


needed = [*SELECTED_MODELS, "parts_mitbersh", *PRIVACY_FILES]
needed += [MODEL_SPECS[k]["config_key"] for k in dict.fromkeys(needed) if "config_key" in MODEL_SPECS.get(k, {})]
needed = list(dict.fromkeys(needed))
FILES = manifest_files()
missing = [k for k in needed if not have(FILES, k)]
if missing:
    print("downloading:", ", ".join(missing))
    with TIMER.step("models.download"):
        proc = run_cmd([sys.executable, "scripts/download_models.py", "--only", ",".join(missing)], check=False)
    if proc.returncode != 0:
        record_error("models", f"scripts/download_models.py exited with {proc.returncode}", keys=",".join(missing))
    FILES = manifest_files()

for key, spec in MODEL_SPECS.items():
    cfg = inference_config(spec.get("config"))
    spec["conf"] = float(cfg.get("conf", DEFAULT_CONF))
    spec["iou"] = float(cfg.get("iou", DEFAULT_IOU))
    spec["conf_source"] = f"{spec['config']} (uncalibrated)" if "conf" in cfg else "uncalibrated default"
    if cfg.get("imgsz") not in (None, spec["imgsz"]):
        print(f"note: {key} config imgsz={cfg.get('imgsz')} differs from the smoke-test imgsz={spec['imgsz']}")

LOADED: dict[str, Any] = {}
MODEL_RECORDS: dict[str, dict[str, Any]] = {}
for key in SELECTED_MODELS:
    spec = MODEL_SPECS[key]
    path = REPO_ROOT / spec["file"]
    entry = FILES.get(key) or {}
    rec: dict[str, Any] = {
        "key": key, "role": spec["role"], "file": spec["file"], "imgsz": spec["imgsz"], "conf": spec["conf"],
        "iou": spec["iou"], "conf_source": spec["conf_source"], "status": "load_failed", "error": None,
        "task": None, "n_classes": None, "names": None, "size_bytes": path.stat().st_size if path.is_file() else None,
        "sha256": entry.get("sha256"), "sha256_ok": None, "source": entry.get("source"),
        "licence_declared": entry.get("licence_declared"),
    }
    try:
        if not path.is_file():
            raise FileNotFoundError(f"{spec['file']} is missing (download failed?)")
        rec["sha256_ok"] = check_sha256(key, path, FILES)
        if rec["sha256_ok"] is False:
            raise ValueError(f"{spec['file']}: sha256 differs from models/manifest.json")
        with TIMER.step("model.load"):
            model = YOLO(str(path))
        names = {int(i): str(n) for i, n in model.names.items()}
        rec.update(status="ok", task=model.task, n_classes=len(names), names=names)
        if "report_classes" in spec:
            spec["classes"] = [i for i, n in names.items() if n in spec["report_classes"]]
            rec["reported_classes"] = [names[i] for i in spec["classes"]]
        LOADED[key] = model
    except Exception as exc:
        rec["error"] = exc_text(exc)
        record_error("models", exc, model=key)
    MODEL_RECORDS[key] = rec

# A separate parts-model instance for the privacy step (plate class); None if it cannot be loaded.
PRIVACY_PARTS = None
_parts_path = REPO_ROOT / MODEL_SPECS["parts_mitbersh"]["file"]
try:
    if not _parts_path.is_file():
        raise FileNotFoundError(f"{MODEL_SPECS['parts_mitbersh']['file']} is missing")
    if check_sha256("parts_mitbersh", _parts_path, FILES) is False:
        raise ValueError("parts model sha256 differs from models/manifest.json")
    PRIVACY_PARTS = YOLO(str(_parts_path))
except Exception as exc:
    record_error("models", exc, model="parts_mitbersh (privacy plate detector)")

PRIVACY_COCO = None  # person-head backstop for faces YuNet misses (privacy, independent of AVTO_MODELS)
_coco_path = REPO_ROOT / MODEL_SPECS["coco_yolo26n"]["file"]
try:
    if not _coco_path.is_file():
        raise FileNotFoundError(f"{MODEL_SPECS['coco_yolo26n']['file']} is missing")
    if check_sha256("coco_yolo26n", _coco_path, FILES) is False:
        raise ValueError("COCO model sha256 differs from models/manifest.json")
    PRIVACY_COCO = YOLO(str(_coco_path))
except Exception as exc:
    record_error("models", exc, model="coco_yolo26n (privacy person-head detector)")

if ENV["yunet"]["status"] == "pending":  # YuNet file was downloaded above: retry the smoke test
    ENV["yunet"] = yunet_smoke()
    write_json(OUT / "env.json", ENV)
    print("YuNet:", ENV["yunet"])

print_table(list(MODEL_RECORDS.values()), ["key", "status", "task", "n_classes", "imgsz", "conf", "iou",
                                           "conf_source", "sha256_ok", "error"])
print("privacy parts-model instance:", "ok" if PRIVACY_PARTS is not None else "unavailable")
print("privacy COCO instance:", "ok" if PRIVACY_COCO is not None else "unavailable")

## 4. Inputs + privacy first

Nothing is displayed or saved in this cell. The anonymiser is built **before** any pixel is decoded:
`Anonymizer(device, extra_detectors=[parts-model plate class + body-panel check, COCO person-head band])`; if
any of these detectors cannot be loaded (plate model, face model, or the parts / COCO models from step 3) the run
stops here. Then, for each video in `data/raw/video/`:
probe, sample frames at
`AVTO_FPS` (full resolution), quality filter (uncalibrated thresholds) -> `OUT/frames_quality.csv`. Every kept frame and
every photo in `data/raw/images/` (full resolution) is anonymised; an input whose anonymisation fails is dropped and
counted (fail closed). Only the anonymised working copies are kept (frames: long side 1280, photos: 1600); the raw
arrays are released at the end of the cell. Per-input privacy reports -> `OUT/privacy.json`.

In [ ]:
# 4. Inputs + privacy. Nothing is displayed or saved here: only anonymised working copies leave this cell.
from avtopassport.io import load_image, resize_max_side
from avtopassport.pipeline.frames import extract_frames, probe_video
from avtopassport.pipeline.privacy import (Anonymizer, PrivacyUnavailable, make_parts_plate_detector,
                                          make_person_head_detector)
from avtopassport.pipeline.quality import DEFAULT_THRESHOLDS, filter_frames

FRAME_SIDE, PHOTO_SIDE = 1280, 1600  # long side of the anonymised working copies
QUALITY_FIELDS = ["video", "index", "t_s", "sharpness", "brightness", "ok", "reason"]


def list_media(sub: str, exts: set[str], cap: int | None) -> list[Path]:
    """Sorted files in data/raw/<sub> with one of `exts`, capped (None = no cap)."""
    folder = RAW / sub
    found = sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in exts) if folder.is_dir() else []
    return found if cap is None else found[:cap]


def privacy_row(name: str, kind: str, report: Any, error: str | None = None) -> dict[str, Any]:
    if report is None:
        return {"input": name, "kind": kind, "ok": False, "n_plates": None, "n_faces": None, "passes": {},
                "errors": [error] if error else []}
    return {"input": name, "kind": kind, "ok": bool(report.ok), "n_plates": int(report.n_plates),
            "n_faces": int(report.n_faces), "passes": {str(k): int(v) for k, v in report.passes.items()},
            "errors": [scrub(str(e)) for e in report.errors]}


def anonymize_or_none(img: np.ndarray, step: str, name: str, kind: str) -> np.ndarray | None:
    """Anonymised copy, or None (input dropped) if any detector failed. The report goes to PRIVACY_ROWS."""
    try:
        with TIMER.step(step):
            out, report = ANON.anonymize(img)
    except Exception as exc:  # anonymize() must not raise; if it does, treat it as a failed detector
        PRIVACY_ROWS.append(privacy_row(name, kind, None, exc_text(exc)))
        record_error("privacy", exc, input=name)
        return None
    PRIVACY_ROWS.append(privacy_row(name, kind, report))
    return out if (out is not None and report.ok) else None


def process_video(vpath: Path) -> None:
    """Probe, sample, quality-filter and anonymise one video. Raw frames stay local to this function, so they are
    released when it returns; only anonymised working copies are appended to BLURRED_FRAMES."""
    vrec: dict[str, Any] = {"video": vpath.name}
    VIDEO_RECORDS.append(vrec)
    try:
        with TIMER.step("video.probe"):
            vrec["probe"] = asdict(probe_video(vpath))
        with TIMER.step("frames.extract"):
            frames, info, frame_errors = extract_frames(vpath, target_fps=FPS, max_frames=MAX_FRAMES)
        with TIMER.step("quality.filter"):
            kept, rejected, qualities = filter_frames(frames)
    except Exception as exc:
        vrec["error"] = exc_text(exc)
        record_error("video", exc, video=vpath.name)
        return
    vrec.update(info=asdict(info), extract_errors=[scrub(str(e)) for e in frame_errors], n_sampled=len(frames),
                n_kept=len(kept), n_rejected=len(rejected), n_blurred=0)
    for fr, q in zip(frames, qualities):
        QUALITY_ROWS.append({"video": vpath.name, "index": fr.index, "t_s": rnd(fr.t_s), "sharpness": rnd(q.sharpness, 2),
                             "brightness": rnd(q.brightness, 2), "ok": bool(q.ok), "reason": q.reason or ""})
    # rejected frames: metadata only, their pixels are never shown or saved
    REJECTED.extend({"video": vpath.name, "index": fr.index, "t_s": rnd(fr.t_s, 2), "reason": q.reason}
                    for fr, q in rejected)
    for fr in kept:
        name = unique_name(f"{vpath.stem}_f{fr.index:05d}", USED_NAMES)
        blurred = anonymize_or_none(fr.image, "privacy.frame", name, "frame")
        if blurred is not None:
            BLURRED_FRAMES.append({"name": name, "kind": "frame", "video": vpath.name, "index": fr.index,
                                   "t_s": fr.t_s, "image": resize_max_side(blurred, FRAME_SIDE)})
            vrec["n_blurred"] += 1


def process_photo(ppath: Path) -> bool:
    """Load one photo at full resolution (in memory only) and anonymise it; False if it could not be loaded."""
    name = unique_name(ppath.stem, USED_NAMES)
    try:
        with TIMER.step("image.load"):
            full = load_image(ppath)
    except Exception as exc:
        record_error("photo", exc, photo=ppath.name)
        return False
    blurred = anonymize_or_none(full, "privacy.photo", name, "photo")
    if blurred is not None:
        BLURRED_PHOTOS.append({"name": name, "kind": "photo", "file": ppath.name,
                               "image": resize_max_side(blurred, PHOTO_SIDE)})
    return True


videos = [] if MAX_FRAMES == 0 else list_media("video", VIDEO_EXTS, None)
photos = list_media("images", IMAGE_EXTS, MAX_IMAGES)

# Fail closed: the anonymiser, with every detector, exists before any pixel is decoded.
if PRIVACY_PARTS is None:
    raise RuntimeError("parts-model plate detector unavailable (see the step 3 errors); stopping: nothing has been "
                       "displayed or saved")
if PRIVACY_COCO is None:
    raise RuntimeError("COCO person-head detector unavailable (see the step 3 errors); stopping: nothing has been "
                       "displayed or saved")
try:
    extra_detectors = [make_parts_plate_detector(PRIVACY_PARTS, device=DEVICE)]
except Exception as exc:
    raise RuntimeError(f"parts-model plate detector unavailable ({exc_text(exc)}); stopping: nothing has been "
                       "displayed or saved") from exc
try:
    extra_detectors.append(make_person_head_detector(PRIVACY_COCO, device=DEVICE))
except Exception as exc:
    raise RuntimeError(f"COCO person-head detector unavailable ({exc_text(exc)}); stopping: nothing has been "
                       "displayed or saved") from exc
try:
    with TIMER.step("privacy.init"):
        ANON = Anonymizer(device=DEVICE, extra_detectors=extra_detectors)
except PrivacyUnavailable as exc:
    raise RuntimeError(f"privacy detectors unavailable ({exc_text(exc)}); stopping: nothing has been displayed or "
                       "saved") from exc

QUALITY_ROWS: list[dict[str, Any]] = []
REJECTED: list[dict[str, Any]] = []  # metadata only: rejected frames are never shown or saved
PRIVACY_ROWS: list[dict[str, Any]] = []
VIDEO_RECORDS: list[dict[str, Any]] = []
BLURRED_FRAMES: list[dict[str, Any]] = []  # anonymised working copies only
BLURRED_PHOTOS: list[dict[str, Any]] = []  # anonymised working copies only
USED_NAMES: set[str] = set()

for vpath in videos:
    process_video(vpath)
photos_processed = sum(process_photo(ppath) for ppath in photos)

frames_kept = sum(r.get("n_kept", 0) for r in VIDEO_RECORDS)
n_photo_load_failed = len(photos) - photos_processed
COUNTS: dict[str, Any] = {
    "videos": len(videos), "frames_sampled": sum(r.get("n_sampled", 0) for r in VIDEO_RECORDS),
    "frames_kept": frames_kept, "frames_rejected": sum(r.get("n_rejected", 0) for r in VIDEO_RECORDS),
    "frames_blurred": len(BLURRED_FRAMES), "frames_dropped_privacy": frames_kept - len(BLURRED_FRAMES),
    "photos": len(photos), "photos_load_failed": n_photo_load_failed, "photos_blurred": len(BLURRED_PHOTOS),
    "photos_dropped_privacy": photos_processed - len(BLURRED_PHOTOS),
    "model_inputs": len(BLURRED_FRAMES) + len(BLURRED_PHOTOS),
}
QUALITY_SUMMARY = {"thresholds": asdict(DEFAULT_THRESHOLDS), "thresholds_basis": "uncalibrated default",
                   "n_frames": len(QUALITY_ROWS), "n_ok": sum(1 for r in QUALITY_ROWS if r["ok"]),
                   "reasons": dict(Counter(r["reason"] for r in QUALITY_ROWS if r["reason"])),
                   "videos": VIDEO_RECORDS}
PRIVACY_SUMMARY = {
    "device": DEVICE, "plate_and_face": "Anonymizer defaults (src/avtopassport/pipeline/privacy.py)",
    "extra_detectors": ["parts_mitbersh license-plate class + body-panel check", "coco_yolo26n person-head band"],
    "n_inputs": len(PRIVACY_ROWS), "n_ok": sum(1 for r in PRIVACY_ROWS if r["ok"]),
    "n_dropped": COUNTS["frames_dropped_privacy"] + COUNTS["photos_dropped_privacy"],
    "n_plates": sum(r["n_plates"] or 0 for r in PRIVACY_ROWS), "n_faces": sum(r["n_faces"] or 0 for r in PRIVACY_ROWS),
    "inputs": PRIVACY_ROWS,
}
write_csv(OUT / "frames_quality.csv", QUALITY_ROWS, QUALITY_FIELDS)
write_json(OUT / "privacy.json", PRIVACY_SUMMARY)
print_table([{"count": k, "value": v} for k, v in COUNTS.items()], ["count", "value"])
print_table([{k: PRIVACY_SUMMARY[k] for k in ("n_inputs", "n_ok", "n_dropped", "n_plates", "n_faces")}],
            ["n_inputs", "n_ok", "n_dropped", "n_plates", "n_faces"])

## 4b. Anonymised frames (display starts here)

Only now is anything shown: a contact sheet of up to 24 **anonymised** kept frames, also saved with
`avtopassport.io.save_jpeg` (no metadata) to `OUT/contact_frames.jpg`. Frames rejected by the quality filter are listed
as a table (index, reason) and never shown.

In [ ]:
# 4b. Display starts here: every array shown or saved below is an anonymised working copy.
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from avtopassport.io import save_jpeg


def contact_sheet(items: Sequence[tuple[str, np.ndarray]], title: str, *, cols: int = 6,
                  max_n: int = 24) -> np.ndarray | None:
    """Show a grid of ANONYMISED BGR images; return it as a BGR array for save_jpeg (None if empty)."""
    items = list(items)[:max_n]
    if not items:
        print(f"{title}: nothing to show")
        return None
    cols = min(cols, len(items))
    rows = math.ceil(len(items) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(2.6 * cols, 3.0 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, (label, image) in zip(axes.flat, items):
        ax.imshow(cv2.cvtColor(resize_max_side(image, 480), cv2.COLOR_BGR2RGB))
        ax.set_title(label, fontsize=7)
    fig.suptitle(title, fontsize=9)
    fig.tight_layout()
    fig.canvas.draw()
    sheet = cv2.cvtColor(np.asarray(fig.canvas.buffer_rgba()), cv2.COLOR_RGBA2BGR)
    plt.show()
    plt.close(fig)
    return sheet


def show_and_save(items: Sequence[tuple[str, np.ndarray]], title: str, path: Path) -> None:
    """contact_sheet() + save_jpeg(); a plotting failure is recorded, never fatal."""
    try:
        sheet = contact_sheet(items, title)
        if sheet is not None:
            save_jpeg(sheet, path, max_side=2400)
    except Exception as exc:
        record_error("display", exc, file=path.name)


shown = [BLURRED_FRAMES[i] for i in evenly_spaced(len(BLURRED_FRAMES), 24)]
show_and_save([(f"{b['video']} #{b['index']} t={b['t_s']:.1f}s", b["image"]) for b in shown],
              f"{RUN}: anonymised kept frames ({len(shown)} of {len(BLURRED_FRAMES)})", OUT / "contact_frames.jpg")
if REJECTED:
    print(f"{len(REJECTED)} frame(s) rejected by the quality filter (uncalibrated thresholds), listed, never shown:")
    display(pd.DataFrame(REJECTED))
else:
    print("no frames rejected by the quality filter")

## 5. Inference

Every loaded damage / parts / COCO model predicts on every anonymised working frame and photo (`device`, `imgsz`,
`conf`, `iou`, `retina_masks=False`, `save=False`). Per prediction: `r.speed` (preprocess + inference + postprocess)
and wall time. Outputs: `OUT/detections.csv` (one row per detection: box in working-copy pixels, mask area in pixels
and as a fraction of the image), annotated images (`r.plot()`, BGR) for every photo and up to 12 frames per model in
`OUT/annotated/<model>/`, a per-model annotated contact sheet `OUT/contact_<model>.jpg`, and a per-model summary
(detections per class, mean confidence, peak CUDA memory). A failing prediction is recorded and the run continues.

**Mapping preview** (damage_mitbersh x parts_mitbersh on the same input): for each damage mask, the part mask with the
largest intersection, as a fraction of the damage area -> `OUT/mapping_preview.csv`. It is a preview only; Phase 3
implements the mapping.

In [ ]:
# 5. Inference on the anonymised working copies -> detections.csv, annotated/, contact sheets, mapping preview.
import time

from ultralytics.utils import ops as yolo_ops

INPUTS = BLURRED_FRAMES + BLURRED_PHOTOS
ANNOTATE = {BLURRED_FRAMES[i]["name"] for i in evenly_spaced(len(BLURRED_FRAMES), 12)} | {b["name"] for b in BLURRED_PHOTOS}
MAP_DAMAGE, MAP_PARTS = "damage_mitbersh", "parts_mitbersh"
DET_FIELDS = ["input", "kind", "model", "class_name", "conf", "x1", "y1", "x2", "y2", "mask_area_px",
              "mask_frac_of_image"]
MAP_FIELDS = ["input", "kind", "damage_class", "damage_conf", "damage_area_px", "part_class", "part_conf",
              "overlap_frac_of_damage", "status"]
DET_ROWS: list[dict[str, Any]] = []
DET_SUMMARY: list[dict[str, Any]] = []
MASKS: dict[tuple[str, str], list[tuple[str, float, np.ndarray]]] = {}  # (model, input) -> bit-packed masks
SHAPES: dict[str, tuple[int, int]] = {}


def predict_kwargs(spec: dict[str, Any], device: str) -> dict[str, Any]:
    kw = dict(device=device, imgsz=spec["imgsz"], conf=spec["conf"], iou=spec["iou"], retina_masks=False,
              save=False, verbose=False)
    if spec.get("classes") is not None:
        kw["classes"] = spec["classes"]
    return kw


def full_res_masks(r: Any) -> np.ndarray | None:
    """(N, H, W) bool masks at the working-copy resolution; None for detect models / no masks."""
    if getattr(r, "masks", None) is None:
        return None
    h, w = r.orig_shape
    try:
        return (yolo_ops.scale_masks(r.masks.data[None].float(), (h, w))[0] > 0.5).cpu().numpy()
    except Exception:  # fallback: rasterise ultralytics' polygons, already in image coordinates
        out = np.zeros((len(r.masks.xy), h, w), bool)
        for i, poly in enumerate(r.masks.xy):
            if len(poly) >= 3:
                layer = np.zeros((h, w), np.uint8)
                cv2.fillPoly(layer, [np.round(poly).astype(np.int32)], 1)
                out[i] = layer.astype(bool)
        return out


if not INPUTS:
    print("no anonymised inputs: inference skipped")
for key, model in (LOADED.items() if INPUTS else ()):
    spec = MODEL_SPECS[key]
    kw = predict_kwargs(spec, DEVICE)
    mem_base = 0
    if IS_CUDA:  # peak memory is reported above what earlier models left allocated
        mem_base = torch.cuda.memory_allocated()
        torch.cuda.reset_peak_memory_stats()
    e2e_ms, wall_ms, confs, per_class = [], [], [], Counter()
    n_ok = n_err = 0
    thumbs: list[tuple[str, np.ndarray]] = []
    for item in INPUTS:
        try:
            t0 = time.perf_counter()
            with TIMER.step(f"infer.{key}"):
                r = model.predict(item["image"], **kw)[0]
            wall_ms.append((time.perf_counter() - t0) * 1000)
            sp = r.speed
            e2e_ms.append(sp["preprocess"] + sp["inference"] + sp["postprocess"])
            h, w = r.orig_shape
            masks = full_res_masks(r)
            n = 0 if r.boxes is None else len(r.boxes)
            xyxy = r.boxes.xyxy.cpu().numpy() if n else np.zeros((0, 4))
            det_conf = r.boxes.conf.cpu().numpy() if n else np.zeros(0)
            det_cls = r.boxes.cls.cpu().numpy().astype(int) if n else np.zeros(0, int)
            labels = [str(r.names[int(c)]) for c in det_cls]
            for j in range(n):
                area = int(masks[j].sum()) if masks is not None and j < len(masks) else None
                DET_ROWS.append({"input": item["name"], "kind": item["kind"], "model": key, "class_name": labels[j],
                                 "conf": rnd(det_conf[j], 4), "x1": rnd(xyxy[j, 0], 1), "y1": rnd(xyxy[j, 1], 1),
                                 "x2": rnd(xyxy[j, 2], 1), "y2": rnd(xyxy[j, 3], 1), "mask_area_px": area,
                                 "mask_frac_of_image": rnd(area / (h * w), 6) if area is not None else None})
                per_class[labels[j]] += 1
                confs.append(float(det_conf[j]))
            if key in (MAP_DAMAGE, MAP_PARTS):
                MASKS[(key, item["name"])] = ([] if masks is None else
                                              [(lab, float(c), np.packbits(m)) for lab, c, m in zip(labels, det_conf, masks)])
                SHAPES[item["name"]] = (h, w)
            if item["name"] in ANNOTATE:
                annotated = r.plot()  # BGR, drawn on the anonymised working copy
                save_jpeg(annotated, OUT / "annotated" / key / f"{item['name']}.jpg", max_side=1600)
                thumbs.append((f"{item['name']} ({n})", resize_max_side(annotated, 480)))
            n_ok += 1
        except Exception as exc:
            n_err += 1
            record_error("inference", exc, model=key, input=item["name"])
    DET_SUMMARY.append({
        "model": key, "device": DEVICE, "imgsz": spec["imgsz"], "conf": spec["conf"], "iou": spec["iou"],
        "inputs": len(INPUTS), "frames": len(BLURRED_FRAMES), "photos": len(BLURRED_PHOTOS), "ok": n_ok,
        "errors": n_err, "detections": sum(per_class.values()), "per_class": dict(sorted(per_class.items())),
        "mean_conf": rnd(mean(confs), 4), "e2e_median_ms": rnd(med(e2e_ms)), "wall_median_ms": rnd(med(wall_ms)),
        "max_mem_mb": rnd((torch.cuda.max_memory_allocated() - mem_base) / 2**20, 1) if IS_CUDA else None,
    })
    show_and_save(thumbs, f"{RUN}: {key} on anonymised inputs (conf {spec['conf']}, uncalibrated)",
                  OUT / f"contact_{key}.jpg")


def map_preview(item: dict[str, Any]) -> list[dict[str, Any]]:
    """For each damage mask on this input, the part mask with the largest intersection / damage area."""
    damage, parts = MASKS.get((MAP_DAMAGE, item["name"])), MASKS.get((MAP_PARTS, item["name"]))
    if not damage or parts is None:
        return []
    h, w = SHAPES[item["name"]]
    part_masks = [(lab, c, np.unpackbits(p, count=h * w).reshape(h, w).astype(bool)) for lab, c, p in parts]
    rows = []
    for d_label, d_conf, packed in damage:
        d_mask = np.unpackbits(packed, count=h * w).reshape(h, w).astype(bool)
        area = int(d_mask.sum())
        best_label, best_conf, best_inter = "", None, 0
        for p_label, p_conf, p_mask in part_masks:
            inter = int(np.count_nonzero(d_mask & p_mask))
            if inter > best_inter:
                best_label, best_conf, best_inter = p_label, p_conf, inter
        rows.append({"input": item["name"], "kind": item["kind"], "damage_class": d_label,
                     "damage_conf": rnd(d_conf, 4), "damage_area_px": area, "part_class": best_label,
                     "part_conf": rnd(best_conf, 4), "overlap_frac_of_damage": rnd(best_inter / area, 4) if area else None,
                     "status": "preview (Phase 3 implements mapping)"})
    return rows


MAP_ROWS: list[dict[str, Any]] = []
for item in INPUTS:
    try:
        MAP_ROWS.extend(map_preview(item))
    except Exception as exc:
        record_error("mapping", exc, input=item["name"])
MASKS.clear()
MAPPING_SUMMARY = {
    "status": "preview (Phase 3 implements mapping)", "damage_model": MAP_DAMAGE, "parts_model": MAP_PARTS,
    "available": MAP_DAMAGE in LOADED and MAP_PARTS in LOADED, "n_rows": len(MAP_ROWS),
    "pairs": [{"damage_class": d, "part_class": p or "(none)", "n": n}
              for (d, p), n in sorted(Counter((r["damage_class"], r["part_class"]) for r in MAP_ROWS).items())],
}
write_csv(OUT / "detections.csv", DET_ROWS, DET_FIELDS)
write_csv(OUT / "mapping_preview.csv", MAP_ROWS, MAP_FIELDS)
display(pd.DataFrame(DET_SUMMARY))
print(f"mapping preview: {len(MAP_ROWS)} damage mask(s) mapped")

## 6. VIN

For each photo in `data/raw/vin/` (`AVTO_MAX_VINS`): load at full resolution **in memory only** (never displayed or
saved) -> `read_vin` (RapidOCR + our validation) -> `validate` on the best raw OCR window -> `decode` (our cited WMI
table + NHTSA vPIC, timeout 4 s, no cache file so the VIN is never written). If `data/raw/vin/ground_truth.csv`
(columns `file,vin`) exists, each photo is matched by file name, else by stem (case-insensitive): exact match and
character errors (Hamming distance over 17 characters); unmatched ground-truth rows are counted. With `AVTO_EASYOCR=1`
EasyOCR (`allowlist` = VIN alphabet) runs as a comparison: its lines, in reading order, go through `read_vin`'s own
windows, validation, ranking and status rules (`avtopassport.vin.ocr._vin_from_lines`), so the two engines differ
only in the recogniser.

Every VIN string printed or stored is masked after position 11 (`first 11 characters + ******`), and validation
corrections that name a character after position 11 are shown as `pos N: *->*`. `OUT/vin_results.json` holds masked
VINs, booleans (exact match, check digit, status), OCR reasons and the decode fields only; all output in this cell
passes the unmasked-VIN guard.

In [ ]:
# 6. VIN: OCR -> validate -> decode. VIN photos are never shown or saved; VINs are masked after position 11.
from avtopassport.vin.decode import decode, known_wmis
from avtopassport.vin.ocr import REASON_PREFIX, make_engine, read_vin
from avtopassport.vin.validate import validate

VIN_ALLOWLIST = "ABCDEFGHJKLMNPRSTUVWXYZ0123456789"
VIN_COLS = ["file", "engine", "status", "vin_masked", "validation_status", "check_digit_ok", "exact_match",
            "char_errors"]
DECODE_KEEP = ("wmi", "manufacturer", "country", "make", "model", "model_year", "model_year_candidates", "body_class",
               "fuel_type", "plant_country", "plant_city", "sources", "vpic_error_code", "vpic_error_text", "notes")
vin_files = list_media("vin", IMAGE_EXTS, MAX_VINS)
GT_PATH = RAW / "vin" / "ground_truth.csv"
GROUND_TRUTH = ground_truth_map(read_csv(GT_PATH))  # lower-case file name -> VIN; secret, never written or printed
add_secret(*GROUND_TRUTH.values())
GT_UNMATCHED = len(set(GROUND_TRUTH) - {gt_key(GROUND_TRUTH, p.name) for p in list_media("vin", IMAGE_EXTS, None)})


def compare(read: str | None, truth: str | None) -> dict[str, Any]:
    if not truth:
        return {"has_ground_truth": False, "exact_match": None, "char_errors": None}
    return {"has_ground_truth": True, "exact_match": bool(read) and read == truth, "char_errors": hamming(read, truth)}


ENGINE, WMIS = None, frozenset()
if vin_files:
    try:
        with TIMER.step("vin.engine_init"):
            ENGINE = make_engine()
        WMIS = known_wmis()
    except Exception as exc:
        record_error("vin", exc, step="engine / WMI table")
READER, EASY_STATUS, vin_from_lines = None, "disabled (AVTO_EASYOCR=0)", None
if EASYOCR and vin_files:
    try:
        import easyocr

        # read_vin's windows / validation / ranking / status applied to another engine's text lines
        from avtopassport.vin.ocr import _vin_from_lines as vin_from_lines

        with TIMER.step("vin.easyocr_init"):
            READER = easyocr.Reader(["en"], gpu=IS_CUDA, verbose=False)
        EASY_STATUS = "ok"
    except Exception as exc:
        READER, EASY_STATUS = None, f"unavailable: {exc_text(exc)}"
        record_error("vin", exc, step="easyocr init")


def ocr_record(name: str, engine: str, res: Any, truth: str | None) -> tuple[dict[str, Any], Any]:
    """Record of one engine's VinOcrResult (secrets remembered, tail corrections redacted) and the validation of
    its best raw window (timed as vin.validate). The caller scrubs the record before it leaves the cell."""
    for cand in res.candidates:
        add_secret(cand.text, cand.validation.raw, cand.validation.vin, *cand.validation.candidates)
    best, val = res.best, None
    if best is not None:
        with TIMER.step("vin.validate"):
            val = validate(best.validation.raw, known_wmis=WMIS)  # the raw window: I/O/Q fixes stay in corrections
        add_secret(val.raw, val.vin, *val.candidates)
    final = val.vin if val is not None else None
    ocr_reasons = [t[len(REASON_PREFIX):] for t, _ in res.raw_lines if str(t).startswith(REASON_PREFIX)]
    rec = {"file": name, "engine": engine, "status": res.status, "vin_masked": mask_vin(final),
           "validation_status": val.status if val else None, "check_digit_ok": val.check_digit_ok if val else None,
           "check_digit_required": val.check_digit_required if val else None,
           "corrections": [redact_tail(c) for c in val.corrections] if val else [],
           "reasons": [redact_tail(r) for r in val.reasons] if val else [],
           "model_year_candidates": list(val.model_year_candidates) if val else [],
           "source": best.source if best is not None else None,
           "ocr_score": rnd(best.score, 4) if best is not None else None, "ocr_reasons": ocr_reasons,
           "n_candidates": len(res.candidates), "candidates_masked": [mask_vin(c.text) for c in res.candidates[:5]],
           "timings_ms": {str(k): rnd(v, 1) for k, v in res.timings_ms.items()}, **compare(final, truth)}
    return rec, val


def rapidocr_vin(vin_img: np.ndarray, name: str, truth: str | None) -> dict[str, Any]:
    """read_vin -> validate -> decode for one in-memory VIN photo; returns a masked, scrubbed record."""
    rec: dict[str, Any] = {"file": name, "engine": "rapidocr", "status": "error"}
    try:
        if ENGINE is None:
            raise RuntimeError("RapidOCR engine unavailable")
        with TIMER.step("vin.ocr"):
            res = read_vin(vin_img, engine=ENGINE, known_wmis=WMIS)
        rec, val = ocr_record(name, "rapidocr", res, truth)
        if val is not None and val.vin and val.status in ("valid", "valid_unverified"):
            with TIMER.step("vin.decode"):
                dec = decode(val.vin, use_vpic=True, timeout=4.0, cache_dir=None)  # no cache file: VIN never written
            dec_dict = asdict(dec)
            rec["decode"] = {k: dec_dict.get(k) for k in DECODE_KEEP}
        else:
            rec["decode"], rec["decode_skipped"] = None, "no valid VIN to decode"
    except Exception as exc:
        rec["error"] = exc_text(exc)
        record_error("vin", exc, file=name)
    return scrub_obj(rec)


def easyocr_vin(vin_img: np.ndarray, name: str, truth: str | None) -> dict[str, Any]:
    """EasyOCR comparison: lines in reading order -> read_vin's post-processing; returns a masked, scrubbed record."""
    rec: dict[str, Any] = {"file": name, "engine": "easyocr", "status": "error"}
    try:
        with TIMER.step("vin.easyocr"):
            found = READER.readtext(vin_img, allowlist=VIN_ALLOWLIST)
        lines = [(str(found[i][1]), float(found[i][2])) for i in reading_order([f[0] for f in found])]
        add_secret(*(t for t, _ in lines), "".join(re.sub(r"[^A-Za-z0-9]", "", t) for t, _ in lines))
        res = vin_from_lines(lines, known_wmis=WMIS, prefix="easyocr", engine="easyocr")
        rec, _ = ocr_record(name, "easyocr", res, truth)
    except Exception as exc:
        rec["error"] = exc_text(exc)
        record_error("vin", exc, file=name, engine="easyocr")
    return scrub_obj(rec)


VIN_RESULTS: list[dict[str, Any]] = []
EASY_RESULTS: list[dict[str, Any]] = []
for vpath in vin_files:
    gt_entry = gt_key(GROUND_TRUTH, vpath.name)
    truth = GROUND_TRUTH[gt_entry] if gt_entry else None
    try:
        with TIMER.step("vin.load"):
            vin_img = load_image(vpath)  # full resolution, in memory only; never shown or saved
    except Exception as exc:
        record_error("vin", exc, file=vpath.name)
        VIN_RESULTS.append(scrub_obj({"file": vpath.name, "engine": "rapidocr", "status": "error",
                                      "error": exc_text(exc)}))
        continue
    VIN_RESULTS.append(rapidocr_vin(vin_img, vpath.name, truth))
    if READER is not None:
        EASY_RESULTS.append(easyocr_vin(vin_img, vpath.name, truth))
    del vin_img
GROUND_TRUTH.clear()  # the guard keeps the secrets (SECRET_VINS) for the rest of the run
truth = None

COUNTS["vin_images"] = len(vin_files)
VIN_SUMMARY = scrub_obj({
    "note": "VINs masked after position 11; unmasked VINs are never written", "n_images": len(vin_files),
    "ground_truth_file": GT_PATH.is_file(), "ground_truth_unmatched_rows": GT_UNMATCHED, "easyocr": EASY_STATUS,
    "n_read": sum(1 for r in VIN_RESULTS if r.get("status") == "read"),
    "n_exact": sum(1 for r in VIN_RESULTS if r.get("exact_match")),
    "easyocr_n_read": sum(1 for r in EASY_RESULTS if r.get("status") == "read"),
    "easyocr_n_exact": sum(1 for r in EASY_RESULTS if r.get("exact_match")),
    "results": VIN_RESULTS, "easyocr_results": EASY_RESULTS,
})
write_json(OUT / "vin_results.json", VIN_SUMMARY)
print_table(VIN_RESULTS + EASY_RESULTS, VIN_COLS)

## 7. Latency (only with `AVTO_LATENCY=1`)

A fixed list of `AVTO_LAT_FRAMES` anonymised working frames (saved to `OUT/latency_frames/` for the CPU benchmark,
then read back so every tier sees the same pixels). Tiers:

* `gpu`: torch on CUDA, every loaded model;
* `cpu_torch`: torch on CPU with `os.cpu_count()` threads, every loaded model;
* `cpu_ort16`, `cpu_ort2` (generally `cpu_ortN`): ONNX Runtime with N threads via `scripts/bench_cpu.py` in its own
  process, deploy candidates only: damage_mitbersh@896, parts_mitbersh@768 (`models/onnx/<key>_<imgsz>.onnx`,
  exported once if missing) and plate_n@640 (`models/onnx/plate_n_640.onnx`, else the downloaded plate ONNX that the
  CPU anonymiser runs).
  `cpu_ort2` is pinned with `taskset` to one physical core (cpu0 and its SMT sibling, `0,8` on this Ryzen) when
  `taskset` exists and there are >= 16 CPUs: the local proxy for a 2-vCPU host, an **optimistic** lower bound.

Protocol (in-process tiers, same as the bench script): one untimed setup prediction, `AVTO_LAT_WARMUP` warm-ups, then
up to `AVTO_LAT_RUNS` timed single-image predictions cycling the frames; the timed phase stops early once it exceeds
`AVTO_LAT_CAP_S` (`cap_hit`). End-to-end ms = preprocess + inference + postprocess from `r.speed`; wall ms is
recorded too. All rows go to `OUT/latency.csv` (columns shared with the bench script).

In [ ]:
# 7. Latency tiers (AVTO_LATENCY=1) -> latency.csv, latency_frames/
LAT_FIELDS = ["run", "tier", "model", "imgsz", "device", "threads", "pinned", "n_warmup", "n_runs", "median_ms",
              "p90_ms", "infer_median_ms", "wall_median_ms", "cap_hit", "loadavg_1m", "protocol", "error"]
LAT_PATH, LAT_DIR = OUT / "latency.csv", OUT / "latency_frames"
BENCH = REPO_ROOT / "scripts" / "bench_cpu.py"
ORT_CANDIDATES = (  # deploy candidates for the ONNX Runtime tiers
    {"key": "damage_mitbersh", "imgsz": 896, "task": "segment", "pt": MODEL_SPECS["damage_mitbersh"]["file"],
     "onnx": "models/onnx/damage_mitbersh_896.onnx"},
    {"key": "parts_mitbersh", "imgsz": 768, "task": "segment", "pt": MODEL_SPECS["parts_mitbersh"]["file"],
     "onnx": "models/onnx/parts_mitbersh_768.onnx"},
    {"key": "plate_n", "imgsz": 640, "task": "detect", "pt": None, "onnx": "models/onnx/plate_n_640.onnx",
     "fallback": "models/plate/license-plate-finetune-v1n.onnx"},  # the downloaded file the CPU anonymiser runs
)
LAT_SKIPPED: list[dict[str, str]] = []


def loadavg_1m() -> float | None:
    try:
        return os.getloadavg()[0]
    except (AttributeError, OSError):
        return None


def pin_cpus() -> str | None:
    """CPU list for the pinned 2-thread tier (cpu0 + its SMT sibling), or None if pinning is not possible."""
    if not shutil.which("taskset") or (os.cpu_count() or 0) < 16:
        return None
    try:
        siblings = Path("/sys/devices/system/cpu/cpu0/topology/thread_siblings_list").read_text().strip()
    except OSError:
        siblings = ""
    return siblings if re.fullmatch(r"\d+[,-]\d+", siblings) else "0,8"


def ensure_onnx(cand: dict[str, Any]) -> Path:
    """The candidate's ONNX file; else its downloaded fallback; else exported once from its .pt (static, simplified)."""
    dest = REPO_ROOT / cand["onnx"]
    if dest.is_file():
        return dest
    if cand.get("fallback") and (REPO_ROOT / cand["fallback"]).is_file():
        return REPO_ROOT / cand["fallback"]
    if not cand["pt"]:
        raise FileNotFoundError(f"{cand['onnx']} is missing")
    with TIMER.step("onnx.export"):
        exported = YOLO(str(REPO_ROOT / cand["pt"])).export(format="onnx", imgsz=cand["imgsz"], simplify=True,
                                                             dynamic=False, device="cpu")
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(exported), str(dest))
    return dest


def lat_row(tier: str, key: str, imgsz: int, device: str, threads: int, *, pinned: bool = False,
            res: dict[str, Any] | None = None, protocol: str = "", error: str = "") -> dict[str, Any]:
    ok = res is not None and res["n_runs"] > 0 and not error
    return {"run": RUN, "tier": tier, "model": key, "imgsz": imgsz, "device": device, "threads": threads,
            "pinned": pinned, "n_warmup": res["n_warmup"] if res else 0, "n_runs": res["n_runs"] if res else 0,
            "median_ms": f3(res["median_ms"]) if ok else "", "p90_ms": f3(res["p90_ms"]) if ok else "",
            "infer_median_ms": f3(res["infer_median_ms"]) if ok else "",
            "wall_median_ms": f3(res["wall_median_ms"]) if ok else "", "cap_hit": bool(res and res["cap_hit"]),
            "loadavg_1m": f2(res["loadavg_1m"] if res else loadavg_1m()), "protocol": protocol,
            "error": error or ("" if ok else "no timed runs completed")}


def bench_inprocess(model: Any, imgs: list[np.ndarray], kw: dict[str, Any],
                    threads: int | None = None) -> dict[str, Any]:
    """Setup call, LAT_WARMUP warm-ups, then up to LAT_RUNS timed predicts; the timed phase stops after LAT_CAP_S.
    `threads` is re-applied after the setup call because ultralytics' select_device("cpu") resets torch to
    min(8, cpu_count - 1) threads when it creates the predictor."""
    t0 = time.perf_counter()
    model.predict(imgs[0], **kw)  # setup: predictor + first prediction, untimed
    load_ms = (time.perf_counter() - t0) * 1000
    if threads:
        torch.set_num_threads(threads)
    for i in range(LAT_WARMUP):
        model.predict(imgs[i % len(imgs)], **kw)
    load_1m = loadavg_1m()
    total, infer, wall = [], [], []
    cap_hit = False
    t_start = time.perf_counter()
    for j in range(LAT_RUNS):
        a = time.perf_counter()
        sp = model.predict(imgs[j % len(imgs)], **kw)[0].speed
        wall.append((time.perf_counter() - a) * 1000)
        total.append(sp["preprocess"] + sp["inference"] + sp["postprocess"])
        infer.append(sp["inference"])
        if time.perf_counter() - t_start > LAT_CAP_S and j + 1 < LAT_RUNS:
            cap_hit = True
            break
    return {"n_warmup": LAT_WARMUP, "n_runs": len(total), "median_ms": med(total), "p90_ms": pctl(total, 0.9),
            "infer_median_ms": med(infer), "wall_median_ms": med(wall), "cap_hit": cap_hit, "loadavg_1m": load_1m,
            "load_ms": load_ms, "threads": torch.get_num_threads()}


def run_torch_tier(tier: str, imgs: list[np.ndarray]) -> None:
    device = (DEVICE if IS_CUDA else "cuda:0") if tier == "gpu" else "cpu"
    want = (os.cpu_count() or 1) if tier == "cpu_torch" else None  # gpu: torch's own CPU thread count
    previous = torch.get_num_threads()
    rows = []
    try:
        for key in LOADED:
            spec = MODEL_SPECS[key]
            proto = "; ".join([
                "e2e-ultralytics", f"in-process torch on {device}", f"warmup={LAT_WARMUP}", f"runs<={LAT_RUNS}",
                f"cap_s={LAT_CAP_S:g} (timed phase)", f"frames={len(imgs)}", f"imgsz={spec['imgsz']}",
                f"conf={spec['conf']}", f"iou={spec['iou']}", "retina_masks=False",
                "median/p90=preprocess+inference+postprocess (r.speed); infer=r.speed inference; wall=predict()"])
            try:
                with TIMER.step(f"latency.{tier}"):
                    res = bench_inprocess(YOLO(str(REPO_ROOT / spec["file"])), imgs, predict_kwargs(spec, device),
                                          threads=want)
                rows.append(lat_row(tier, key, spec["imgsz"], device, res["threads"], res=res,
                                    protocol=f"{proto}; torch_threads={res['threads']} (timed phase); "
                                             f"load_ms={res['load_ms']:.1f} (untimed)"))
            except Exception as exc:
                rows.append(lat_row(tier, key, spec["imgsz"], device, want or torch.get_num_threads(), protocol=proto,
                                    error=exc_text(exc)))
                record_error("latency", exc, tier=tier, model=key)
            finally:
                if device.startswith("cuda"):
                    torch.cuda.empty_cache()
    finally:
        torch.set_num_threads(previous)
        if rows:
            append_csv(LAT_PATH, rows, LAT_FIELDS)


def run_ort_tier(tier: str, threads: int) -> None:
    pin = pin_cpus() if tier == "cpu_ort2" else None
    env = {**os.environ, "YOLO_AUTOINSTALL": "false", "CUDA_VISIBLE_DEVICES": ""}
    for cand in ORT_CANDIDATES:
        if cand["key"] in ALL_MODEL_KEYS and cand["key"] not in SELECTED_MODELS:
            continue
        try:
            onnx_path = ensure_onnx(cand)
        except Exception as exc:
            record_error("latency", exc, tier=tier, model=cand["key"], step="onnx export")
            append_csv(LAT_PATH, [lat_row(tier, cand["key"], cand["imgsz"], "cpu", threads, pinned=bool(pin),
                                          protocol="onnx export", error=exc_text(exc))], LAT_FIELDS)
            continue
        cmd = [sys.executable, str(BENCH), "--onnx", str(onnx_path), "--model-key", cand["key"],
               "--imgsz", str(cand["imgsz"]), "--task", cand["task"], "--frames-dir", str(LAT_DIR),
               "--threads", str(threads), "--warmup", str(LAT_WARMUP), "--runs", str(LAT_RUNS),
               "--cap-s", str(LAT_CAP_S), "--run", RUN, "--pinned", "1" if pin else "0", "--out", str(LAT_PATH)]
        if pin:
            cmd = ["taskset", "-c", pin, *cmd]
        try:
            with TIMER.step(f"latency.{tier}"):
                proc = run_cmd(cmd, check=False, env=env, cwd=REPO_ROOT, timeout=4 * LAT_CAP_S + 900)
            if proc.returncode != 0:
                record_error("latency", f"bench_cpu.py exited with {proc.returncode}", tier=tier, model=cand["key"])
        except Exception as exc:  # timeout, taskset / interpreter not runnable
            record_error("latency", exc, tier=tier, model=cand["key"])


if not LATENCY:
    print("AVTO_LATENCY=0: latency tiers skipped")
else:
    pool = BLURRED_FRAMES or BLURRED_PHOTOS
    lat_imgs: list[np.ndarray] = []
    try:
        lat_paths = [save_jpeg(pool[i]["image"], LAT_DIR / f"lat_{n:02d}.jpg", max_side=None)
                     for n, i in enumerate(evenly_spaced(len(pool), LAT_FRAMES))]
        lat_imgs = [load_image(p) for p in lat_paths]  # every tier sees the same (JPEG-decoded) pixels
    except Exception as exc:
        record_error("latency", exc, step="latency frames")
    if not lat_imgs:
        LAT_SKIPPED.append({"tier": "all", "reason": "no anonymised frames or photos for latency"})
    else:
        for tier in LAT_TIERS:
            if tier in ("gpu", "cpu_torch"):
                if torch is None or (tier == "gpu" and not CUDA_OK):
                    LAT_SKIPPED.append({"tier": tier, "reason": "torch / CUDA not available"})
                elif not LOADED:
                    LAT_SKIPPED.append({"tier": tier, "reason": "no model loaded"})
                else:
                    run_torch_tier(tier, lat_imgs)
                continue
            threads = int(tier[len("cpu_ort"):])
            if not BENCH.is_file():
                LAT_SKIPPED.append({"tier": tier, "reason": "scripts/bench_cpu.py is missing"})
            elif threads > (os.cpu_count() or 1):
                LAT_SKIPPED.append({"tier": tier, "reason": f"{threads} threads > os.cpu_count()={os.cpu_count()}"})
            else:
                run_ort_tier(tier, threads)
LAT_ROWS = [r for r in read_csv(LAT_PATH) if r.get("run") == RUN]
for s in LAT_SKIPPED:
    print(f"skipped tier {s['tier']}: {s['reason']}")
if LAT_ROWS:
    display(pd.DataFrame(LAT_ROWS)[["tier", "model", "imgsz", "threads", "pinned", "n_runs", "median_ms", "p90_ms",
                                    "wall_median_ms", "cap_hit", "error"]])

## 8. Per-inspection estimate (only if this run has latency rows)

For each latency tier: estimate = sum over pipeline steps of (measured median x count in **this** run: videos, kept
frames, photos, VIN photos). Model steps use the tier's end-to-end latency medians (damage_mitbersh and
parts_mitbersh); decode, quality, privacy, OCR and vPIC use this run's `StepTimer` medians (measured in this notebook
process on this run's device with the libraries' default thread pools, stated in the `source` and `note` columns: for
`cpu_ortN` tiers those steps did not run on N threads, so the total is optimistic). Included and excluded steps are
listed explicitly. For `cpu_ort2` a column `x2 safety factor (assumption)` doubles each figure. Written to
`OUT/estimate.csv`, labelled "derived from measured medians".

In [ ]:
# 8. Per-inspection estimate -> estimate.csv (derived from measured medians)
EST_ROWS: list[dict[str, Any]] = []
if not LAT_ROWS:
    print("no latency rows for this run: estimate skipped (run with AVTO_LATENCY=1)")
else:
    EST_COUNTS = {"videos": COUNTS["videos"], "kept_frames": COUNTS["frames_kept"], "photos": COUNTS["photos"],
                  "model_inputs": COUNTS["model_inputs"], "vins": COUNTS.get("vin_images", 0)}
    EST_ROWS = build_estimate(RUN, LAT_ROWS, TIMER.summary(), EST_COUNTS, device=DEVICE, cpu_count=os.cpu_count())
    write_csv(OUT / "estimate.csv", EST_ROWS, EST_FIELDS)
    print("counts used:", EST_COUNTS)
    display(pd.DataFrame(EST_ROWS)[["tier", "step", "included", "count", "median_ms", "subtotal_ms",
                                    "x2 safety factor (assumption)", "note"]])

## 9. Summary

`OUT/results.json` (keys: `schema_version` (1), `run`, `created_at`, `notebook`, `config`, `env`, `models`, `counts`,
`quality`, `privacy`, `detections`, `mapping_preview`, `vin` (masked), `latency`, `latency_skipped`, `estimate`,
`timings` (StepTimer summary), `errors`, `artifacts`; every string VIN-scrubbed before writing) and `OUT/summary.md`
(generated markdown tables).
`scripts/render_results.py` turns every run's results into the tables in `docs/research/smoke_test_results.md`.
On Colab the outputs folder is zipped and offered for download.

In [ ]:
# 9. Summary -> results.json, summary.md (+ zip download on Colab)
ARTIFACTS = sorted({p.relative_to(OUT).as_posix() for p in OUT.rglob("*") if p.is_file()} | {"results.json", "summary.md"})
RESULTS = scrub_obj({  # every string VIN-scrubbed: a masked copy is written instead of stopping at the guard
    "schema_version": 1, "run": RUN, "created_at": now_iso(), "notebook": "notebooks/01_smoke_test_colab.ipynb",
    "config": CONFIG, "env": ENV, "models": [MODEL_RECORDS[k] for k in sorted(MODEL_RECORDS)], "counts": COUNTS,
    "quality": QUALITY_SUMMARY, "privacy": {k: v for k, v in PRIVACY_SUMMARY.items() if k != "inputs"},
    "detections": DET_SUMMARY, "mapping_preview": MAPPING_SUMMARY, "vin": VIN_SUMMARY, "latency": LAT_ROWS,
    "latency_skipped": LAT_SKIPPED, "estimate": EST_ROWS, "timings": TIMER.summary(), "errors": ERRORS,
    "artifacts": ARTIFACTS,
})
write_json(OUT / "results.json", RESULTS)
SUMMARY_MD = render_summary_md(RESULTS)
write_text(OUT / "summary.md", SUMMARY_MD)
display(Markdown(guard(SUMMARY_MD)))
print(f"{len(ERRORS)} error(s); outputs in {OUT.relative_to(REPO_ROOT)}")
if IN_COLAB:
    from google.colab import files

    archive = shutil.make_archive(str(OUT.parent / f"{RUN}_outputs"), "zip", root_dir=OUT)
    files.download(archive)